# ArcAGI2 — ARC Prize 2026 (ARC-AGI-2 code track) submission notebook

Self-contained: the whole solver core is embedded below (official submissions
run without internet, so nothing is cloned or piped at submit time).

**Run order**: Cell 1 (discovery) -> cells 2-8 (write modules) -> cell 9
(self-check) -> cell 10 (solve + submission.json).

**Legend**: ✅ = verified against the official rules (2026-10-18);
⚠️ = adaptation point; fill from cell 1's printed layout.

The solver is a closed transform DSL with a consistency filter: a program is
emitted only when it reproduces EVERY demonstration pair exactly, so a wrong
program can never be submitted by accident. Attempts are diverse (simplest
consistent program + next-simplest with a different family signature), which
matches the official either-attempt scoring.

Reference: https://github.com/Zhiyilang/arcagi2 (MIT)

In [ ]:
# CELL 1 — DISCOVERY: classify every JSON under /kaggle/input, then describe.
# ✅ Verified (2026-10-18): a task file is JSON with keys "train" and "test";
#    train entries have input+output, test entries have input (+output in the
#    public evaluation set, absent in the hidden test set).
# ⚠️ The dataset root varies per competition snapshot — discovered below.
# ⚠️ 2026-10-24 Kaggle live-run fix: a *solutions* file
#    ({task_id: [expected test outputs]}) used to be misread as a multi-task
#    file because the old `all(... if isinstance(v, dict))` filter skipped the
#    list-valued entries and an empty generator makes all() True. Classify
#    first, index second — never assume a layout.
import os, json

def classify(d):
    # single / multi / solutions / list / other — see cell 10 for the loader
    if isinstance(d, dict):
        if "train" in d and "test" in d:
            return "single"
        if d and all(isinstance(v, dict) and "train" in v and "test" in v
                     for v in d.values()):
            return "multi"
        if d and all(isinstance(v, list) for v in d.values()):
            return "solutions"
        return "other"
    if isinstance(d, list):
        if d and all(isinstance(x, dict) and "train" in x and "test" in x
                     for x in d):
            return "list"
        return "other"
    return "other"

kinds = {}
if os.path.isdir("/kaggle/input"):
    for dirpath, _dirs, files in os.walk("/kaggle/input"):
        for fn in sorted(files):
            if not fn.endswith(".json"):
                continue
            p = os.path.join(dirpath, fn)
            try:
                if os.path.getsize(p) > 40 * 1024 * 1024:
                    continue
                with open(p, "r", encoding="utf-8") as fh:
                    d = json.load(fh)
            except Exception:
                continue
            kinds.setdefault(classify(d), []).append(p)

for kind in sorted(kinds):
    print("== %-9s %d file(s)" % (kind, len(kinds[kind])))
    for p in kinds[kind][:10]:
        size = os.path.getsize(p)
        try:
            with open(p, "r", encoding="utf-8") as fh:
                d = json.load(fh)
        except Exception:
            print("   %s (%d B, unreadable)" % (p, size))
            continue
        if kind == "single":
            print("   %s (%d B) train=%d test=%d test[0] keys=%s"
                  % (p, size, len(d["train"]), len(d["test"]),
                     sorted(d["test"][0].keys())))
        elif kind == "multi":
            k = sorted(d)[0]
            print("   %s (%d B) tasks=%d first=%s train=%d test=%d"
                  % (p, size, len(d), k,
                     len(d[k]["train"]), len(d[k]["test"])))
        elif kind == "solutions":
            print("   %s (%d B) expected outputs for %d tasks (NOT a task "
                  "file — cell 10 uses it only to score locally)"
                  % (p, size, len(d)))
        elif kind == "list":
            print("   %s (%d B) %d tasks in a list (ids assigned by position)"
                  % (p, size, len(d)))
        else:
            print("   %s (%d B)" % (p, size))
print("DISCOVERY DONE — cell 10 loads single/multi/list files only")

# %%writefile does not create intermediate directories, and Kaggle's
# /kaggle/working starts empty — create the package dir once, here.
# (2026-10-24: the first writefile cell died with FileNotFoundError without
# this; locally the directory already existed, so CI never caught it.)
os.makedirs("/kaggle/working/arcagi2", exist_ok=True)

## Cells 2-8 — materialize the embedded core (offline CI-green modules)

Each cell writes one module of the `arcagi2` package into the notebook working
directory. `python eval.py` on the same code passes locally (also under
`LC_ALL=C` without `PYTHONUTF8`), and the real-data gate keeps the honest
score on the official public evaluation set in the loop.

In [ ]:
import base64 as _b, os as _o
_PATH = "/kaggle/working/arcagi2/grids.py"
_o.makedirs(_o.path.dirname(_PATH), exist_ok=True)
with open(_PATH, "w", encoding="utf-8") as _f:
    _f.write(_b.b64decode("IiIiR3JpZCBwcmltaXRpdmVzIGZvciB0aGUgQVJDLUFHSS0yIHNvbHZlci4KCgoKQSBHcmlkIGlzIGFuIGltbXV0YWJsZSB0dXBsZSBvZiB0dXBsZXMgb2YgaW50cyAoY29sb3JzIDAtOSkuIEV2ZXJ5IGZ1bmN0aW9uIGluCgp0aGlzIG1vZHVsZSBpcyBwdXJlOiBub3RoaW5nIG11dGF0ZXMgaXRzIGlucHV0LCBhbmQgbm90aGluZyBkZXBlbmRzIG9uIGdsb2JhbAoKc3RhdGUuIFRoaXMga2VlcHMgdGhlIHByb2dyYW0gc2VhcmNoIHJlcHJvZHVjaWJsZSBhbmQgdGhlIHRlc3Qgc3VpdGUgaG9uZXN0LgoKIiIiCgoKCmZyb20gY29sbGVjdGlvbnMgaW1wb3J0IENvdW50ZXIKCgoKR3JpZCA9IHR1cGxlICAjIHR1cGxlW3R1cGxlW2ludCwgLi4uXSwgLi4uXQoKCgpCR19GQUxMQkFDSyA9IDAKCgoKCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gY29uc3RydWN0b3JzCgoKCmRlZiBmcm9tX3Jvd3Mocm93cyk6CgogICAgIiIiQnVpbGQgYSBncmlkIGZyb20gYSBsaXN0IG9mIGxpc3RzICh0aGUgSlNPTiB0YXNrIGZvcm1hdCkuIiIiCgogICAgcmV0dXJuIHR1cGxlKHR1cGxlKGludChjKSBmb3IgYyBpbiByb3cpIGZvciByb3cgaW4gcm93cykKCgoKCgpkZWYgdG9fcm93cyhnKToKCiAgICAiIiJJbnZlcnNlIG9mIGZyb21fcm93cyAoZm9yIEpTT04gc2VyaWFsaXphdGlvbikuIiIiCgogICAgcmV0dXJuIFtsaXN0KHJvdykgZm9yIHJvdyBpbiBnXQoKCgoKCmRlZiBibGFuayhoLCB3LCBjb2xvcj0wKToKCiAgICByZXR1cm4gdHVwbGUodHVwbGUoY29sb3IgZm9yIF8gaW4gcmFuZ2UodykpIGZvciBfIGluIHJhbmdlKGgpKQoKCgoKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSBhY2Nlc3NvcnMKCgoKZGVmIHNoYXBlKGcpOgoKICAgIHJldHVybiAobGVuKGcpLCBsZW4oZ1swXSkgaWYgZyBlbHNlIDApCgoKCgoKZGVmIGhlaWdodChnKToKCiAgICByZXR1cm4gbGVuKGcpCgoKCgoKZGVmIHdpZHRoKGcpOgoKICAgIHJldHVybiBsZW4oZ1swXSkgaWYgZyBlbHNlIDAKCgoKCgpkZWYgY29sb3JzX3VzZWQoZyk6CgogICAgb3V0ID0gc2V0KCkKCiAgICBmb3Igcm93IGluIGc6CgogICAgICAgIG91dC51cGRhdGUocm93KQoKICAgIHJldHVybiBvdXQKCgoKCgpkZWYgY29sb3JfY291bnRzKGcpOgoKICAgIGMgPSBDb3VudGVyKCkKCiAgICBmb3Igcm93IGluIGc6CgogICAgICAgIGMudXBkYXRlKHJvdykKCiAgICByZXR1cm4gYwoKCgoKCmRlZiBtb3N0X2NvbW1vbl9jb2xvcihnLCBleGNsdWRlPSgpKToKCiAgICBjID0gY29sb3JfY291bnRzKGcpCgogICAgZm9yIGNvbCBpbiBleGNsdWRlOgoKICAgICAgICBjLnBvcChjb2wsIE5vbmUpCgogICAgaWYgbm90IGM6CgogICAgICAgIHJldHVybiBCR19GQUxMQkFDSwoKICAgIGJlc3RfbiA9IG1heChjLnZhbHVlcygpKQoKICAgIHRpZWQgPSBzb3J0ZWQoY29sIGZvciBjb2wsIG4gaW4gYy5pdGVtcygpIGlmIG4gPT0gYmVzdF9uKQoKICAgICMgQVJDIGNvbnZlbnRpb246IDAgaXMgdGhlIGJhY2tncm91bmQsIHNvIGl0IHdpbnMgY291bnQgdGllcy4KCiAgICByZXR1cm4gMCBpZiAwIGluIHRpZWQgZWxzZSB0aWVkWzBdCgoKCgoKZGVmIGJhY2tncm91bmRfY29sb3IoZyk6CgogICAgIiIiSGV1cmlzdGljIGJhY2tncm91bmQ6IHRoZSBtb3N0IGNvbW1vbiBjb2xvciBpbiB0aGUgZ3JpZC4iIiIKCiAgICByZXR1cm4gbW9zdF9jb21tb25fY29sb3IoZykKCgoKCgpkZWYgY2VsbChnLCByLCBjKToKCiAgICBpZiAwIDw9IHIgPCBoZWlnaHQoZykgYW5kIDAgPD0gYyA8IHdpZHRoKGcpOgoKICAgICAgICByZXR1cm4gZ1tyXVtjXQoKICAgIHJldHVybiBOb25lCgoKCgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gY29tcGFyaXNvbgoKCgpkZWYgZ3JpZHNfZXF1YWwoYSwgYik6CgogICAgcmV0dXJuIGEgPT0gYgoKCgoKCmRlZiBkaWZmX2NlbGxzKGEsIGIpOgoKICAgICIiIkNlbGxzIChyLCBjLCBjb2xvcl9hLCBjb2xvcl9iKSB3aGVyZSB0d28gc2FtZS1zaGFwZWQgZ3JpZHMgZGlmZmVyLiIiIgoKICAgIGlmIHNoYXBlKGEpICE9IHNoYXBlKGIpOgoKICAgICAgICByZXR1cm4gTm9uZQoKICAgIG91dCA9IFtdCgogICAgZm9yIHIgaW4gcmFuZ2UoaGVpZ2h0KGEpKToKCiAgICAgICAgZm9yIGMgaW4gcmFuZ2Uod2lkdGgoYSkpOgoKICAgICAgICAgICAgaWYgYVtyXVtjXSAhPSBiW3JdW2NdOgoKICAgICAgICAgICAgICAgIG91dC5hcHBlbmQoKHIsIGMsIGFbcl1bY10sIGJbcl1bY10pKQoKICAgIHJldHVybiBvdXQKCgoKCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSBnZW9tZXRyeQoKCgpkZWYgcm90YXRlOTAoZyk6CgogICAgcmV0dXJuIHR1cGxlKHppcCgqZ1s6Oi0xXSkpCgoKCgoKZGVmIHJvdGF0ZTE4MChnKToKCiAgICByZXR1cm4gdHVwbGUodHVwbGUocm93KSBmb3Igcm93IGluIHJvdGF0ZTkwKHJvdGF0ZTkwKGcpKSkKCgoKCgpkZWYgcm90YXRlMjcwKGcpOgoKICAgIHJldHVybiB0dXBsZSh6aXAoKmcpKVs6Oi0xXQoKCgoKCmRlZiBmbGlwX2goZyk6CgogICAgIiIiTWlycm9yIGxlZnQtcmlnaHQuIiIiCgogICAgcmV0dXJuIHR1cGxlKHR1cGxlKHJvd1s6Oi0xXSkgZm9yIHJvdyBpbiBnKQoKCgoKCmRlZiBmbGlwX3YoZyk6CgogICAgIiIiTWlycm9yIHRvcC1ib3R0b20uIiIiCgogICAgcmV0dXJuIGdbOjotMV0KCgoKCgpkZWYgdHJhbnNwb3NlKGcpOgoKICAgIHJldHVybiB0dXBsZSh6aXAoKmcpKQoKCgoKCmRlZiBhbnRpX3RyYW5zcG9zZShnKToKCiAgICByZXR1cm4gcm90YXRlMTgwKHRyYW5zcG9zZShnKSkKCgoKCgpkZWYgdHJhbnNsYXRlKGcsIGRoLCBkdywgYmc9QkdfRkFMTEJBQ0spOgoKICAgICIiIlNoaWZ0IGNvbnRlbnQgYnkgKGRoLCBkdyk7IHZhY2F0ZWQgY2VsbHMgYmVjb21lIGJnLiIiIgoKICAgIGgsIHcgPSBzaGFwZShnKQoKICAgIG91dCA9IFtbYmddICogdyBmb3IgXyBpbiByYW5nZShoKV0KCiAgICBmb3IgciBpbiByYW5nZShoKToKCiAgICAgICAgZm9yIGMgaW4gcmFuZ2Uodyk6CgogICAgICAgICAgICBuciwgbmMgPSByICsgZGgsIGMgKyBkdwoKICAgICAgICAgICAgaWYgMCA8PSBuciA8IGggYW5kIDAgPD0gbmMgPCB3OgoKICAgICAgICAgICAgICAgIG91dFtucl1bbmNdID0gZ1tyXVtjXQoKICAgIHJldHVybiB0dXBsZSh0dXBsZShyb3cpIGZvciByb3cgaW4gb3V0KQoKCgoKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gY3JvcC9wYWQKCgoKZGVmIGJib3goZywgYmc9Tm9uZSk6CgogICAgIiIiQm91bmRpbmcgYm94IChyMCwgYzAsIHIxLCBjMSkgb2Ygbm9uLWJnIGNlbGxzOyBOb25lIGlmIGVtcHR5LiIiIgoKICAgIGlmIGJnIGlzIE5vbmU6CgogICAgICAgIGJnID0gYmFja2dyb3VuZF9jb2xvcihnKQoKICAgIHIwLCBjMCwgcjEsIGMxID0gaGVpZ2h0KGcpLCB3aWR0aChnKSwgLTEsIC0xCgogICAgZm9yIHIgaW4gcmFuZ2UoaGVpZ2h0KGcpKToKCiAgICAgICAgZm9yIGMgaW4gcmFuZ2Uod2lkdGgoZykpOgoKICAgICAgICAgICAgaWYgZ1tyXVtjXSAhPSBiZzoKCiAgICAgICAgICAgICAgICByMCA9IG1pbihyMCwgcikKCiAgICAgICAgICAgICAgICBjMCA9IG1pbihjMCwgYykKCiAgICAgICAgICAgICAgICByMSA9IG1heChyMSwgcikKCiAgICAgICAgICAgICAgICBjMSA9IG1heChjMSwgYykKCiAgICBpZiByMSA8IHIwOgoKICAgICAgICByZXR1cm4gTm9uZQoKICAgIHJldHVybiAocjAsIGMwLCByMSwgYzEpCgoKCgoKZGVmIGNyb3BfYmJveChnLCBiZz1Ob25lKToKCiAgICBib3ggPSBiYm94KGcsIGJnKQoKICAgIGlmIGJveCBpcyBOb25lOgoKICAgICAgICByZXR1cm4gZwoKICAgIHIwLCBjMCwgcjEsIGMxID0gYm94CgogICAgcmV0dXJuIHR1cGxlKHR1cGxlKGdbcl1bYzA6YzEgKyAxXSkgZm9yIHIgaW4gcmFuZ2UocjAsIHIxICsgMSkpCgoKCgoKZGVmIHBhZF90byhnLCBoLCB3LCBiZz1CR19GQUxMQkFDSywgYW5jaG9yPSJjZW50ZXIiKToKCiAgICAiIiJQYWQgKG9yIGNyb3ApIHRvIGV4YWN0bHkgKGgsIHcpLiBhbmNob3I6IGNlbnRlcnx0bHx0cnxibHxici4iIiIKCiAgICBjdXJfaCwgY3VyX3cgPSBzaGFwZShnKQoKICAgIGlmIGggPCBjdXJfaCBvciB3IDwgY3VyX3c6CgogICAgICAgIHJldHVybiBjcm9wX3RvKGcsIGgsIHcsIGFuY2hvcikKCiAgICB0b3AsIGxlZnQgPSBfYW5jaG9yX29mZnNldHMoYW5jaG9yLCBjdXJfaCwgY3VyX3csIGgsIHcpCgogICAgb3V0ID0gW1tiZ10gKiB3IGZvciBfIGluIHJhbmdlKGgpXQoKICAgIGZvciByIGluIHJhbmdlKGN1cl9oKToKCiAgICAgICAgZm9yIGMgaW4gcmFuZ2UoY3VyX3cpOgoKICAgICAgICAgICAgb3V0W3RvcCArIHJdW2xlZnQgKyBjXSA9IGdbcl1bY10KCiAgICByZXR1cm4gdHVwbGUodHVwbGUocm93KSBmb3Igcm93IGluIG91dCkKCgoKCgpkZWYgY3JvcF90byhnLCBoLCB3LCBhbmNob3I9ImNlbnRlciIpOgoKICAgIGNoLCBjdyA9IHNoYXBlKGcpCgogICAgdG9wLCBsZWZ0ID0gX2FuY2hvcl9vZmZzZXRzKGFuY2hvciwgY2gsIGN3LCBoLCB3KQoKICAgIG91dCA9IFtdCgogICAgZm9yIHIgaW4gcmFuZ2UodG9wLCB0b3AgKyBoKToKCiAgICAgICAgcm93ID0gW10KCiAgICAgICAgZm9yIGMgaW4gcmFuZ2UobGVmdCwgbGVmdCArIHcpOgoKICAgICAgICAgICAgcm93LmFwcGVuZChnW3JdW2NdIGlmIDAgPD0gciA8IGNoIGFuZCAwIDw9IGMgPCBjdyBlbHNlIEJHX0ZBTExCQUNLKQoKICAgICAgICBvdXQuYXBwZW5kKHR1cGxlKHJvdykpCgogICAgcmV0dXJuIHR1cGxlKG91dCkKCgoKCgpkZWYgX2FuY2hvcl9vZmZzZXRzKGFuY2hvciwgY3VyX2gsIGN1cl93LCBuZXdfaCwgbmV3X3cpOgoKICAgIGlmIGFuY2hvciA9PSAidGwiOgoKICAgICAgICByZXR1cm4gMCwgMAoKICAgIGlmIGFuY2hvciA9PSAidHIiOgoKICAgICAgICByZXR1cm4gMCwgbWF4KDAsIG5ld193IC0gY3VyX3cpCgogICAgaWYgYW5jaG9yID09ICJibCI6CgogICAgICAgIHJldHVybiBtYXgoMCwgbmV3X2ggLSBjdXJfaCksIDAKCiAgICBpZiBhbmNob3IgPT0gImJyIjoKCiAgICAgICAgcmV0dXJuIG1heCgwLCBuZXdfaCAtIGN1cl9oKSwgbWF4KDAsIG5ld193IC0gY3VyX3cpCgogICAgcmV0dXJuIG1heCgwLCAobmV3X2ggLSBjdXJfaCkgLy8gMiksIG1heCgwLCAobmV3X3cgLSBjdXJfdykgLy8gMikgICMgY2VudGVyCgoKCgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tIHJlc2l6ZQoKCgpkZWYgdXBzY2FsZShnLCBmaCwgZncpOgoKICAgIGlmIGZoIDw9IDAgb3IgZncgPD0gMDoKCiAgICAgICAgcmV0dXJuIGcKCiAgICBvdXQgPSBbXQoKICAgIGZvciByb3cgaW4gZzoKCiAgICAgICAgYmlnX3JvdyA9IFtdCgogICAgICAgIGZvciBjZWxsX3YgaW4gcm93OgoKICAgICAgICAgICAgYmlnX3Jvdy5leHRlbmQoW2NlbGxfdl0gKiBmdykKCiAgICAgICAgZm9yIF8gaW4gcmFuZ2UoZmgpOgoKICAgICAgICAgICAgb3V0LmFwcGVuZCh0dXBsZShiaWdfcm93KSkKCiAgICByZXR1cm4gdHVwbGUob3V0KQoKCgoKCmRlZiBkb3duc2NhbGUoZywgZmgsIGZ3LCBtb2RlPSJtYWpvcml0eSIpOgoKICAgICIiIlJlZHVjZSBlYWNoIGZoIHggZncgYmxvY2sgdG8gb25lIGNlbGwuCgoKCiAgICBtb2RlOgoKICAgICAgbWFqb3JpdHkgLSBtb3N0IGNvbW1vbiBjb2xvciBpbiB0aGUgYmxvY2sgKHRpZXMgLT4gc21hbGxlciBjb2xvcikKCiAgICAgIGZpcnN0ICAgIC0gdG9wLWxlZnQgY2VsbAoKICAgICAgdW5pcXVlICAgLSB0aGUgY29sb3IgaWYgdGhlIGJsb2NrIGlzIHVuaWZvcm0sIGVsc2UgbWFqb3JpdHkKCiAgICAiIiIKCiAgICBoLCB3ID0gc2hhcGUoZykKCiAgICBpZiBmaCA8PSAwIG9yIGZ3IDw9IDA6CgogICAgICAgIHJldHVybiBnCgogICAgb2gsIG93ID0gaCAvLyBmaCwgdyAvLyBmdwoKICAgIGlmIG9oID09IDAgb3Igb3cgPT0gMDoKCiAgICAgICAgcmV0dXJuIGcKCiAgICBvdXQgPSBbXQoKICAgIGZvciBiciBpbiByYW5nZShvaCk6CgogICAgICAgIHJvdyA9IFtdCgogICAgICAgIGZvciBiYyBpbiByYW5nZShvdyk6CgogICAgICAgICAgICBibG9jayA9IFsKCiAgICAgICAgICAgICAgICBnW2JyICogZmggKyBkcl1bYmMgKiBmdyArIGRjXQoKICAgICAgICAgICAgICAgIGZvciBkciBpbiByYW5nZShmaCkKCiAgICAgICAgICAgICAgICBmb3IgZGMgaW4gcmFuZ2UoZncpCgogICAgICAgICAgICBdCgogICAgICAgICAgICBpZiBtb2RlID09ICJmaXJzdCI6CgogICAgICAgICAgICAgICAgcm93LmFwcGVuZChibG9ja1swXSkKCiAgICAgICAgICAgIGVsaWYgbW9kZSA9PSAidW5pcXVlIjoKCiAgICAgICAgICAgICAgICByb3cuYXBwZW5kKGJsb2NrWzBdIGlmIGxlbihzZXQoYmxvY2spKSA9PSAxIGVsc2UKCiAgICAgICAgICAgICAgICAgICAgICAgICAgIENvdW50ZXIoYmxvY2spLm1vc3RfY29tbW9uKDEpWzBdWzBdKQoKICAgICAgICAgICAgZWxzZTogICMgbWFqb3JpdHkKCiAgICAgICAgICAgICAgICBjbnQgPSBDb3VudGVyKGJsb2NrKQoKICAgICAgICAgICAgICAgIGJlc3QgPSBtYXgoY250Lml0ZW1zKCksIGtleT1sYW1iZGEga3Y6IChrdlsxXSwgLWt2WzBdKSlbMF0KCiAgICAgICAgICAgICAgICByb3cuYXBwZW5kKGJlc3QpCgogICAgICAgIG91dC5hcHBlbmQodHVwbGUocm93KSkKCiAgICByZXR1cm4gdHVwbGUob3V0KQoKCgoKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gY29sb3Igb3BzCgoKCmRlZiByZWNvbG9yKGcsIG1hcHBpbmcpOgoKICAgIHJldHVybiB0dXBsZSh0dXBsZShtYXBwaW5nLmdldCh2LCB2KSBmb3IgdiBpbiByb3cpIGZvciByb3cgaW4gZykKCgoKCgpkZWYga2VlcF9jb2xvcnMoZywgY29sb3JzLCBiZz1CR19GQUxMQkFDSyk6CgogICAga2VlcCA9IHNldChjb2xvcnMpCgogICAgcmV0dXJuIHR1cGxlKHR1cGxlKHYgaWYgdiBpbiBrZWVwIGVsc2UgYmcgZm9yIHYgaW4gcm93KSBmb3Igcm93IGluIGcpCgoKCgoKZGVmIHJlcGxhY2VfY29sb3IoZywgb2xkLCBuZXcpOgoKICAgIHJldHVybiB0dXBsZSh0dXBsZShuZXcgaWYgdiA9PSBvbGQgZWxzZSB2IGZvciB2IGluIHJvdykgZm9yIHJvdyBpbiBnKQoKCgoKCmRlZiBzd2FwX2NvbG9ycyhnLCBhLCBiKToKCiAgICByZXR1cm4gdHVwbGUoCgogICAgICAgIHR1cGxlKGIgaWYgdiA9PSBhIGVsc2UgKGEgaWYgdiA9PSBiIGVsc2UgdikgZm9yIHYgaW4gcm93KSBmb3Igcm93IGluIGcKCiAgICApCgoKCgoKZGVmIGludmVydF9jb2xvcnMoZywgcGFsZXR0ZSk6CgogICAgIiIiTWFwIGVhY2ggY29sb3IgdG8gaXRzIGluZGV4LW1pcnJvciBpbnNpZGUgdGhlIGdpdmVuIHBhbGV0dGUuIiIiCgogICAgcGFsID0gbGlzdChwYWxldHRlKQoKICAgIG1hcHBpbmcgPSB7YzogcGFsW2xlbihwYWwpIC0gMSAtIGldIGZvciBpLCBjIGluIGVudW1lcmF0ZShwYWwpfQoKICAgIHJldHVybiByZWNvbG9yKGcsIG1hcHBpbmcpCgoKCgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gc3RydWN0dXJhbAoKCgpkZWYgdGlsZShnLCBuaCwgbncpOgoKICAgIG91dCA9IFtdCgogICAgZm9yIF8gaW4gcmFuZ2UobmgpOgoKICAgICAgICBmb3Igcm93IGluIGc6CgogICAgICAgICAgICBvdXQuYXBwZW5kKHR1cGxlKHJvdykgKiBudykKCiAgICByZXR1cm4gdHVwbGUob3V0KQoKCgoKCmRlZiBmaWxsX2hvbGVzKGcsIGhvbGU9MCwgY29sb3I9Tm9uZSk6CgogICAgIiIiRmlsbCByZWdpb25zIG9mIGBob2xlYCBjb2xvciB0aGF0IGFyZSBmdWxseSBlbmNsb3NlZCBieSBvdGhlciBjb2xvcnMuCgoKCiAgICBjb2xvciBkZWZhdWx0cyB0byB0aGUgbW9zdCBjb21tb24gbm9uLWhvbGUgY29sb3IgKHRoZSB1c3VhbCBBUkMgaW50ZW50OgoKICAgICJmaWxsIHRoZSBlbmNsb3NlZCByZWdpb24gd2l0aCB0aGUgc3Vycm91bmRpbmcgY29sb3IiKS4KCiAgICAiIiIKCiAgICBoLCB3ID0gc2hhcGUoZykKCiAgICBvdXRzaWRlID0gW1tGYWxzZV0gKiB3IGZvciBfIGluIHJhbmdlKGgpXQoKICAgIHN0YWNrID0gW10KCiAgICBmb3IgciBpbiByYW5nZShoKToKCiAgICAgICAgZm9yIGMgaW4gcmFuZ2Uodyk6CgogICAgICAgICAgICBpZiAociBpbiAoMCwgaCAtIDEpIG9yIGMgaW4gKDAsIHcgLSAxKSkgYW5kIGdbcl1bY10gPT0gaG9sZSBhbmQgbm90IG91dHNpZGVbcl1bY106CgogICAgICAgICAgICAgICAgb3V0c2lkZVtyXVtjXSA9IFRydWUKCiAgICAgICAgICAgICAgICBzdGFjay5hcHBlbmQoKHIsIGMpKQoKICAgIHdoaWxlIHN0YWNrOgoKICAgICAgICByLCBjID0gc3RhY2sucG9wKCkKCiAgICAgICAgZm9yIGRyLCBkYyBpbiAoKDEsIDApLCAoLTEsIDApLCAoMCwgMSksICgwLCAtMSkpOgoKICAgICAgICAgICAgbnIsIG5jID0gciArIGRyLCBjICsgZGMKCiAgICAgICAgICAgIGlmIDAgPD0gbnIgPCBoIGFuZCAwIDw9IG5jIDwgdyBhbmQgbm90IG91dHNpZGVbbnJdW25jXSBhbmQgZ1tucl1bbmNdID09IGhvbGU6CgogICAgICAgICAgICAgICAgb3V0c2lkZVtucl1bbmNdID0gVHJ1ZQoKICAgICAgICAgICAgICAgIHN0YWNrLmFwcGVuZCgobnIsIG5jKSkKCiAgICBpZiBjb2xvciBpcyBOb25lOgoKICAgICAgICBjb3VudHMgPSB7fQoKICAgICAgICBmb3Igcm93IGluIGc6CgogICAgICAgICAgICBmb3IgdiBpbiByb3c6CgogICAgICAgICAgICAgICAgaWYgdiAhPSBob2xlOgoKICAgICAgICAgICAgICAgICAgICBjb3VudHNbdl0gPSBjb3VudHMuZ2V0KHYsIDApICsgMQoKICAgICAgICBpZiBub3QgY291bnRzOgoKICAgICAgICAgICAgcmV0dXJuIGcKCiAgICAgICAgY29sb3IgPSBtYXgoY291bnRzLml0ZW1zKCksIGtleT1sYW1iZGEga3Y6IChrdlsxXSwgLWt2WzBdKSlbMF0KCiAgICBvdXQgPSBbXQoKICAgIGZvciByIGluIHJhbmdlKGgpOgoKICAgICAgICByb3cgPSBbXQoKICAgICAgICBmb3IgYyBpbiByYW5nZSh3KToKCiAgICAgICAgICAgIHJvdy5hcHBlbmQoY29sb3IgaWYgKGdbcl1bY10gPT0gaG9sZSBhbmQgbm90IG91dHNpZGVbcl1bY10pCgogICAgICAgICAgICAgICAgICAgICAgIGVsc2UgZ1tyXVtjXSkKCiAgICAgICAgb3V0LmFwcGVuZCh0dXBsZShyb3cpKQoKICAgIHJldHVybiB0dXBsZShvdXQpCgoKCgoKZGVmIGNvbXBsZXRlX2hfc3ltbWV0cnkoZywgYmc9Tm9uZSk6CgogICAgIiIiTWlycm9yLWNvbXBsZXRlIGxlZnQtcmlnaHQ6IGZpbGwgYmcgY2VsbHMgZnJvbSB0aGVpciBtaXJyb3IgaWYgc2V0LiIiIgoKICAgIGlmIGJnIGlzIE5vbmU6CgogICAgICAgIGJnID0gYmFja2dyb3VuZF9jb2xvcihnKQoKICAgIHcgPSB3aWR0aChnKQoKICAgIG91dCA9IFtdCgogICAgZm9yIHJvdyBpbiBnOgoKICAgICAgICBuZXcgPSBsaXN0KHJvdykKCiAgICAgICAgZm9yIGMgaW4gcmFuZ2Uodyk6CgogICAgICAgICAgICBtID0gdyAtIDEgLSBjCgogICAgICAgICAgICBpZiBuZXdbY10gPT0gYmcgYW5kIG5ld1ttXSAhPSBiZzoKCiAgICAgICAgICAgICAgICBuZXdbY10gPSBuZXdbbV0KCiAgICAgICAgICAgIGVsaWYgbmV3W21dID09IGJnIGFuZCBuZXdbY10gIT0gYmc6CgogICAgICAgICAgICAgICAgbmV3W21dID0gbmV3W2NdCgogICAgICAgIG91dC5hcHBlbmQodHVwbGUobmV3KSkKCiAgICByZXR1cm4gdHVwbGUob3V0KQoKCgoKCmRlZiBjb21wbGV0ZV92X3N5bW1ldHJ5KGcsIGJnPU5vbmUpOgoKICAgIHJldHVybiB0cmFuc3Bvc2UoY29tcGxldGVfaF9zeW1tZXRyeSh0cmFuc3Bvc2UoZyksIGJnKSkKCgoKCgpkZWYgZ3Jhdml0eShnLCBkaXJlY3Rpb24sIGJnPU5vbmUpOgoKICAgICIiIlNsaWRlIGV2ZXJ5IG5vbi1iZyBjZWxsIGluIGBkaXJlY3Rpb25gIHVudGlsIGJsb2NrZWQuIiIiCgogICAgaWYgYmcgaXMgTm9uZToKCiAgICAgICAgYmcgPSBiYWNrZ3JvdW5kX2NvbG9yKGcpCgogICAgaCwgdyA9IHNoYXBlKGcpCgogICAgaWYgZGlyZWN0aW9uID09ICJkb3duIjoKCiAgICAgICAgZm9yIGMgaW4gcmFuZ2Uodyk6CgogICAgICAgICAgICB3cml0ZSA9IGggLSAxCgogICAgICAgICAgICBmb3IgciBpbiByYW5nZShoIC0gMSwgLTEsIC0xKToKCiAgICAgICAgICAgICAgICBpZiBnW3JdW2NdICE9IGJnOgoKICAgICAgICAgICAgICAgICAgICBnID0gX3NldChnLCB3cml0ZSwgYywgZ1tyXVtjXSkgaWYgd3JpdGUgIT0gciBlbHNlIGcKCiAgICAgICAgICAgICAgICAgICAgd3JpdGUgLT0gMQoKICAgICAgICAgICAgZm9yIHIgaW4gcmFuZ2Uod3JpdGUsIC0xLCAtMSk6CgogICAgICAgICAgICAgICAgZyA9IF9zZXQoZywgciwgYywgYmcpCgogICAgZWxpZiBkaXJlY3Rpb24gPT0gInVwIjoKCiAgICAgICAgZm9yIGMgaW4gcmFuZ2Uodyk6CgogICAgICAgICAgICB3cml0ZSA9IDAKCiAgICAgICAgICAgIGZvciByIGluIHJhbmdlKGgpOgoKICAgICAgICAgICAgICAgIGlmIGdbcl1bY10gIT0gYmc6CgogICAgICAgICAgICAgICAgICAgIGcgPSBfc2V0KGcsIHdyaXRlLCBjLCBnW3JdW2NdKSBpZiB3cml0ZSAhPSByIGVsc2UgZwoKICAgICAgICAgICAgICAgICAgICB3cml0ZSArPSAxCgogICAgICAgICAgICBmb3IgciBpbiByYW5nZSh3cml0ZSwgaCk6CgogICAgICAgICAgICAgICAgZyA9IF9zZXQoZywgciwgYywgYmcpCgogICAgZWxpZiBkaXJlY3Rpb24gPT0gInJpZ2h0IjoKCiAgICAgICAgZm9yIHIgaW4gcmFuZ2UoaCk6CgogICAgICAgICAgICB3cml0ZSA9IHcgLSAxCgogICAgICAgICAgICBmb3IgYyBpbiByYW5nZSh3IC0gMSwgLTEsIC0xKToKCiAgICAgICAgICAgICAgICBpZiBnW3JdW2NdICE9IGJnOgoKICAgICAgICAgICAgICAgICAgICBnID0gX3NldChnLCByLCB3cml0ZSwgZ1tyXVtjXSkgaWYgd3JpdGUgIT0gYyBlbHNlIGcKCiAgICAgICAgICAgICAgICAgICAgd3JpdGUgLT0gMQoKICAgICAgICAgICAgZm9yIGMgaW4gcmFuZ2Uod3JpdGUsIC0xLCAtMSk6CgogICAgICAgICAgICAgICAgZyA9IF9zZXQoZywgciwgYywgYmcpCgogICAgZWxpZiBkaXJlY3Rpb24gPT0gImxlZnQiOgoKICAgICAgICBmb3IgciBpbiByYW5nZShoKToKCiAgICAgICAgICAgIHdyaXRlID0gMAoKICAgICAgICAgICAgZm9yIGMgaW4gcmFuZ2Uodyk6CgogICAgICAgICAgICAgICAgaWYgZ1tyXVtjXSAhPSBiZzoKCiAgICAgICAgICAgICAgICAgICAgZyA9IF9zZXQoZywgciwgd3JpdGUsIGdbcl1bY10pIGlmIHdyaXRlICE9IGMgZWxzZSBnCgogICAgICAgICAgICAgICAgICAgIHdyaXRlICs9IDEKCiAgICAgICAgICAgIGZvciBjIGluIHJhbmdlKHdyaXRlLCB3KToKCiAgICAgICAgICAgICAgICBnID0gX3NldChnLCByLCBjLCBiZykKCiAgICByZXR1cm4gZwoKCgoKCmRlZiBfc2V0KGcsIHIsIGMsIHYpOgoKICAgIHJldHVybiBnWzpyXSArIChnW3JdWzpjXSArICh2LCkgKyBnW3JdW2MgKyAxOl0sKSArIGdbciArIDE6XQoKCgoKCmRlZiBib3JkZXIoZywgY29sb3IsIHRoaWNrbmVzcz0xKToKCiAgICBoLCB3ID0gc2hhcGUoZykKCiAgICB0ID0gbWluKHRoaWNrbmVzcywgaCAtIDEsIHcgLSAxKQoKICAgIGlmIHQgPCAxOgoKICAgICAgICByZXR1cm4gZwoKICAgIG91dCA9IFtsaXN0KHJvdykgZm9yIHJvdyBpbiBnXQoKICAgIGZvciBrIGluIHJhbmdlKHQpOgoKICAgICAgICBmb3IgYyBpbiByYW5nZSh3KToKCiAgICAgICAgICAgIG91dFtrXVtjXSA9IGNvbG9yCgogICAgICAgICAgICBvdXRbaCAtIDEgLSBrXVtjXSA9IGNvbG9yCgogICAgICAgIGZvciByIGluIHJhbmdlKGgpOgoKICAgICAgICAgICAgb3V0W3JdW2tdID0gY29sb3IKCiAgICAgICAgICAgIG91dFtyXVt3IC0gMSAtIGtdID0gY29sb3IKCiAgICByZXR1cm4gdHVwbGUodHVwbGUocm93KSBmb3Igcm93IGluIG91dCkKCgoKCgpkZWYgb3V0bGluZV9ub25iZyhnLCBjb2xvciwgYmc9Tm9uZSk6CgogICAgIiIiU2V0IHRvIGBjb2xvcmAgZXZlcnkgYmcgY2VsbCBvcnRob2dvbmFsbHkgYWRqYWNlbnQgdG8gYSBub24tYmcgY2VsbC4iIiIKCiAgICBpZiBiZyBpcyBOb25lOgoKICAgICAgICBiZyA9IGJhY2tncm91bmRfY29sb3IoZykKCiAgICBoLCB3ID0gc2hhcGUoZykKCiAgICBvdXQgPSBbbGlzdChyb3cpIGZvciByb3cgaW4gZ10KCiAgICBmb3IgciBpbiByYW5nZShoKToKCiAgICAgICAgZm9yIGMgaW4gcmFuZ2Uodyk6CgogICAgICAgICAgICBpZiBnW3JdW2NdID09IGJnIGFuZCBfaGFzX25vbmJnX25laWdoYm9yKGcsIHIsIGMsIGJnKToKCiAgICAgICAgICAgICAgICBvdXRbcl1bY10gPSBjb2xvcgoKICAgIHJldHVybiB0dXBsZSh0dXBsZShyb3cpIGZvciByb3cgaW4gb3V0KQoKCgoKCmRlZiBfaGFzX25vbmJnX25laWdoYm9yKGcsIHIsIGMsIGJnKToKCiAgICBmb3IgZHIsIGRjIGluICgoMSwgMCksICgtMSwgMCksICgwLCAxKSwgKDAsIC0xKSk6CgogICAgICAgIHYgPSBjZWxsKGcsIHIgKyBkciwgYyArIGRjKQoKICAgICAgICBpZiB2IGlzIG5vdCBOb25lIGFuZCB2ICE9IGJnOgoKICAgICAgICAgICAgcmV0dXJuIFRydWUKCiAgICByZXR1cm4gRmFsc2U=").decode("utf-8"))


In [ ]:
import base64 as _b, os as _o
_PATH = "/kaggle/working/arcagi2/objects.py"
_o.makedirs(_o.path.dirname(_PATH), exist_ok=True)
with open(_PATH, "w", encoding="utf-8") as _f:
    _f.write(_b.b64decode("IiIiQ29ubmVjdGVkLWNvbXBvbmVudCBvYmplY3RzIGZvciB0aGUgQVJDLUFHSS0yIHNvbHZlci4KCgoKT2JqZWN0cyBhcmUgZXh0cmFjdGVkIGZyb20gYSBncmlkIGFzIG1heGltYWwgc2FtZS1jb2xvciBjb25uZWN0ZWQgcmVnaW9ucwoKKDQtIG9yIDgtY29ubmVjdGl2aXR5KS4gVGhlIG9iamVjdCBsYXllciBpcyB3aGF0IGxldHMgdGhlIHNvbHZlciBleHByZXNzCgoibW92ZSB0aGUgcmVkIHNxdWFyZSBvbnRvIHRoZSBibHVlIGxpbmUiIHdpdGhvdXQgaGFuZC1jb2RlZCBjb29yZGluYXRlcy4KCiIiIgoKCgpmcm9tIGNvbGxlY3Rpb25zIGltcG9ydCBkZXF1ZQoKCgpmcm9tIC4gaW1wb3J0IGdyaWRzCgpmcm9tIC5ncmlkcyBpbXBvcnQgR3JpZCwgc2hhcGUsIGNlbGwsIGJsYW5rLCBoZWlnaHQsIHdpZHRoCgoKCgoKY2xhc3MgT2JqOgoKICAgICIiIkEgY29ubmVjdGVkIGNvbXBvbmVudC4gSW1tdXRhYmxlIHNuYXBzaG90IG9mIGNlbGxzICsgZGVyaXZlZCBmYWN0cy4iIiIKCgoKICAgIF9fc2xvdHNfXyA9ICgiY29sb3IiLCAiY2VsbHMiLCAiYmJveCIsICJiZyIpCgoKCiAgICBkZWYgX19pbml0X18oc2VsZiwgY29sb3IsIGNlbGxzLCBiZyk6CgogICAgICAgIHNlbGYuY29sb3IgPSBjb2xvcgoKICAgICAgICBzZWxmLmNlbGxzID0gZnJvemVuc2V0KGNlbGxzKQoKICAgICAgICBycyA9IFtyIGZvciByLCBfIGluIHNlbGYuY2VsbHNdCgogICAgICAgIGNzID0gW2MgZm9yIF8sIGMgaW4gc2VsZi5jZWxsc10KCiAgICAgICAgc2VsZi5iYm94ID0gKG1pbihycyksIG1pbihjcyksIG1heChycyksIG1heChjcykpCgogICAgICAgIHNlbGYuYmcgPSBiZwoKCgogICAgQHByb3BlcnR5CgogICAgZGVmIHNpemUoc2VsZik6CgogICAgICAgIHJldHVybiBsZW4oc2VsZi5jZWxscykKCgoKICAgIEBwcm9wZXJ0eQoKICAgIGRlZiBoKHNlbGYpOgoKICAgICAgICByZXR1cm4gc2VsZi5iYm94WzJdIC0gc2VsZi5iYm94WzBdICsgMQoKCgogICAgQHByb3BlcnR5CgogICAgZGVmIHcoc2VsZik6CgogICAgICAgIHJldHVybiBzZWxmLmJib3hbM10gLSBzZWxmLmJib3hbMV0gKyAxCgoKCiAgICBAcHJvcGVydHkKCiAgICBkZWYgY2VudGVyKHNlbGYpOgoKICAgICAgICBycyA9IFtyIGZvciByLCBfIGluIHNlbGYuY2VsbHNdCgogICAgICAgIGNzID0gW2MgZm9yIF8sIGMgaW4gc2VsZi5jZWxsc10KCiAgICAgICAgcmV0dXJuIChzdW0ocnMpIC8gbGVuKHJzKSwgc3VtKGNzKSAvIGxlbihjcykpCgoKCiAgICBkZWYgc3ViZ3JpZChzZWxmLCBwYWQ9MCk6CgogICAgICAgICIiIlRoZSBvYmplY3QgcmVuZGVyZWQgYWxvbmUgb24gYSBiYWNrZ3JvdW5kIGNhbnZhcy4iIiIKCiAgICAgICAgcjAsIGMwLCByMSwgYzEgPSBzZWxmLmJib3gKCiAgICAgICAgaCA9IHIxIC0gcjAgKyAxICsgMiAqIHBhZAoKICAgICAgICB3ID0gYzEgLSBjMCArIDEgKyAyICogcGFkCgogICAgICAgIG91dCA9IFtbc2VsZi5iZ10gKiB3IGZvciBfIGluIHJhbmdlKGgpXQoKICAgICAgICBmb3IgciwgYyBpbiBzZWxmLmNlbGxzOgoKICAgICAgICAgICAgb3V0W3IgLSByMCArIHBhZF1bYyAtIGMwICsgcGFkXSA9IHNlbGYuY29sb3IKCiAgICAgICAgcmV0dXJuIHR1cGxlKHR1cGxlKHJvdykgZm9yIHJvdyBpbiBvdXQpCgoKCiAgICBkZWYgbm9ybWFsaXplZF9jZWxscyhzZWxmKToKCiAgICAgICAgIiIiQ2VsbHMgc2hpZnRlZCBzbyB0aGUgYmJveCBjb3JuZXIgaXMgKDAsIDApLiIiIgoKICAgICAgICByMCwgYzAgPSBzZWxmLmJib3hbMF0sIHNlbGYuYmJveFsxXQoKICAgICAgICByZXR1cm4gZnJvemVuc2V0KChyIC0gcjAsIGMgLSBjMCkgZm9yIHIsIGMgaW4gc2VsZi5jZWxscykKCgoKICAgIGRlZiBzYW1lX3NoYXBlX2FzKHNlbGYsIG90aGVyKToKCiAgICAgICAgcmV0dXJuIHNlbGYubm9ybWFsaXplZF9jZWxscygpID09IG90aGVyLm5vcm1hbGl6ZWRfY2VsbHMoKQoKCgogICAgZGVmIF9fcmVwcl9fKHNlbGYpOgoKICAgICAgICByZXR1cm4gIk9iaihjb2xvcj0lcywgc2l6ZT0lZCwgYmJveD0lcykiICUgKAoKICAgICAgICAgICAgc2VsZi5jb2xvciwgc2VsZi5zaXplLCBzZWxmLmJib3gpCgoKCgoKZGVmIGNvbXBvbmVudHMoZywgY29ubmVjdGl2aXR5PTQsIGJnPU5vbmUsIG1pbl9zaXplPTEpOgoKICAgICIiIkFsbCBtYXhpbWFsIGNvbm5lY3RlZCBzYW1lLWNvbG9yIHJlZ2lvbnMgKHRoZSBiZyBjb2xvciBpcyBza2lwcGVkCgogICAgdW5sZXNzIGl0IGlzIHRoZSBvbmx5IGNvbG9yIHByZXNlbnQpLiIiIgoKICAgIGlmIGJnIGlzIE5vbmU6CgogICAgICAgIGJnID0gZ3JpZHMuYmFja2dyb3VuZF9jb2xvcihnKQoKICAgIGgsIHcgPSBzaGFwZShnKQoKICAgIHNlZW4gPSBbW0ZhbHNlXSAqIHcgZm9yIF8gaW4gcmFuZ2UoaCldCgogICAgb3V0ID0gW10KCiAgICBpZiBjb25uZWN0aXZpdHkgPT0gNDoKCiAgICAgICAgc3RlcHMgPSAoKDEsIDApLCAoLTEsIDApLCAoMCwgMSksICgwLCAtMSkpCgogICAgZWxzZToKCiAgICAgICAgc3RlcHMgPSAoKDEsIDApLCAoLTEsIDApLCAoMCwgMSksICgwLCAtMSksCgogICAgICAgICAgICAgICAgICgxLCAxKSwgKDEsIC0xKSwgKC0xLCAxKSwgKC0xLCAtMSkpCgogICAgZm9yIHIgaW4gcmFuZ2UoaCk6CgogICAgICAgIGZvciBjIGluIHJhbmdlKHcpOgoKICAgICAgICAgICAgaWYgc2VlbltyXVtjXSBvciBnW3JdW2NdID09IGJnOgoKICAgICAgICAgICAgICAgIGNvbnRpbnVlCgogICAgICAgICAgICBjb2xvciA9IGdbcl1bY10KCiAgICAgICAgICAgIGNlbGxzID0gW10KCiAgICAgICAgICAgIGRxID0gZGVxdWUoWyhyLCBjKV0pCgogICAgICAgICAgICBzZWVuW3JdW2NdID0gVHJ1ZQoKICAgICAgICAgICAgd2hpbGUgZHE6CgogICAgICAgICAgICAgICAgY3IsIGNjID0gZHEucG9wbGVmdCgpCgogICAgICAgICAgICAgICAgY2VsbHMuYXBwZW5kKChjciwgY2MpKQoKICAgICAgICAgICAgICAgIGZvciBkciwgZGMgaW4gc3RlcHM6CgogICAgICAgICAgICAgICAgICAgIG5yLCBuYyA9IGNyICsgZHIsIGNjICsgZGMKCiAgICAgICAgICAgICAgICAgICAgaWYgMCA8PSBuciA8IGggYW5kIDAgPD0gbmMgPCB3IGFuZCBub3Qgc2Vlbltucl1bbmNdIGFuZCBnW25yXVtuY10gPT0gY29sb3I6CgogICAgICAgICAgICAgICAgICAgICAgICBzZWVuW25yXVtuY10gPSBUcnVlCgogICAgICAgICAgICAgICAgICAgICAgICBkcS5hcHBlbmQoKG5yLCBuYykpCgogICAgICAgICAgICBpZiBsZW4oY2VsbHMpID49IG1pbl9zaXplOgoKICAgICAgICAgICAgICAgIG91dC5hcHBlbmQoT2JqKGNvbG9yLCBjZWxscywgYmcpKQoKICAgIG91dC5zb3J0KGtleT1sYW1iZGEgbzogKG8uYmJveFswXSwgby5iYm94WzFdLCAtby5zaXplKSkKCiAgICByZXR1cm4gb3V0CgoKCgoKZGVmIGNvbXBvbmVudHNfYnlfY29sb3IoZywgY29ubmVjdGl2aXR5PTQsIGJnPU5vbmUpOgoKICAgICIiImNvbXBvbmVudHMoKSwgZ3JvdXBlZCBwZXIgbm9uLWJnIGNvbG9yLCBwcmVzZXJ2aW5nIGRpc2NvdmVyeSBvcmRlci4iIiIKCiAgICBvdXQgPSB7fQoKICAgIGZvciBvIGluIGNvbXBvbmVudHMoZywgY29ubmVjdGl2aXR5LCBiZyk6CgogICAgICAgIG91dC5zZXRkZWZhdWx0KG8uY29sb3IsIFtdKS5hcHBlbmQobykKCiAgICByZXR1cm4gb3V0CgoKCgoKZGVmIHBhaW50KGcsIG9ianMsIGJnPTApOgoKICAgICIiIlBhaW50IG9iamVjdHMgb250byBhIGNvcHkgb2YgZ3JpZCBnIChpbiBwbGFjZSBzZW1hbnRpY3MgdmlhIHJlYnVpbGQpLiIiIgoKICAgIGgsIHcgPSBzaGFwZShnKQoKICAgIG91dCA9IFtsaXN0KHJvdykgZm9yIHJvdyBpbiBnXQoKICAgIGZvciBvIGluIG9ianM6CgogICAgICAgIGZvciByLCBjIGluIG8uY2VsbHM6CgogICAgICAgICAgICBpZiAwIDw9IHIgPCBoIGFuZCAwIDw9IGMgPCB3OgoKICAgICAgICAgICAgICAgIG91dFtyXVtjXSA9IG8uY29sb3IKCiAgICByZXR1cm4gdHVwbGUodHVwbGUocm93KSBmb3Igcm93IGluIG91dCkKCgoKCgpkZWYgbW92ZV9vYmplY3Qob2JqLCBkciwgZGMpOgoKICAgICIiIkEgY29weSBvZiB0aGUgb2JqZWN0IHNoaWZ0ZWQgYnkgKGRyLCBkYykuIiIiCgogICAgcmV0dXJuIE9iaihvYmouY29sb3IsIFsociArIGRyLCBjICsgZGMpIGZvciByLCBjIGluIG9iai5jZWxsc10sIG9iai5iZykKCgoKCgpkZWYgcmVjb2xvcmVkKG9iaiwgY29sb3IpOgoKICAgIHJldHVybiBPYmooY29sb3IsIG9iai5jZWxscywgb2JqLmJnKQoKCgoKCmRlZiBsYXJnZXN0KG9ianMpOgoKICAgIGlmIG5vdCBvYmpzOgoKICAgICAgICByZXR1cm4gTm9uZQoKICAgIHJldHVybiBtYXgob2Jqcywga2V5PWxhbWJkYSBvOiBvLnNpemUpCgoKCgoKZGVmIHNtYWxsZXN0KG9ianMpOgoKICAgIGlmIG5vdCBvYmpzOgoKICAgICAgICByZXR1cm4gTm9uZQoKICAgIHJldHVybiBtaW4ob2Jqcywga2V5PWxhbWJkYSBvOiBvLnNpemUpCgoKCgoKZGVmIHNvcnRfYnkob2Jqcywga2V5KToKCiAgICByZXR1cm4gc29ydGVkKG9ianMsIGtleT1rZXkpCgoKCgoKZGVmIG9iamVjdHNfb3ZlcmxhcChhLCBiKToKCiAgICByZXR1cm4gYm9vbChhLmNlbGxzICYgYi5jZWxscykKCgoKCgpkZWYgZ3JpZF9vZl9vYmplY3RzKG9ianMsIGgsIHcsIGJnPTApOgoKICAgIHJldHVybiBwYWludChibGFuayhoLCB3LCBiZyksIG9ianMsIGJnKQoKCgoKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSBvYmplY3QtbGV2ZWwgbW92ZW1lbnQKCgoKZGVmIF9zaGlmdF9vayhjZWxscywgZHIsIGRjLCBoLCB3LCBvdGhlcnMpOgoKICAgIGZvciByLCBjIGluIGNlbGxzOgoKICAgICAgICBuciwgbmMgPSByICsgZHIsIGMgKyBkYwoKICAgICAgICBpZiBub3QgKDAgPD0gbnIgPCBoIGFuZCAwIDw9IG5jIDwgdyk6CgogICAgICAgICAgICByZXR1cm4gRmFsc2UKCiAgICAgICAgaWYgKG5yLCBuYykgaW4gb3RoZXJzOgoKICAgICAgICAgICAgcmV0dXJuIEZhbHNlCgogICAgcmV0dXJuIFRydWUKCgoKCgpkZWYgb2JqZWN0X2dyYXZpdHkoZywgZGlyZWN0aW9uLCBiZz1Ob25lLCBjb25uZWN0aXZpdHk9NCk6CgogICAgIiIiU2xpZGUgZXZlcnkgY29ubmVjdGVkIGNvbXBvbmVudCBhcyBhIFVOSVQgdW50aWwgYmxvY2tlZC4KCgoKICAgIFVubGlrZSBjZWxsLWxldmVsIGdyYXZpdHksIGFuIG9iamVjdCBrZWVwcyBpdHMgc2hhcGUgd2hpbGUgZmFsbGluZy4KCiAgICBPYmplY3RzIGFyZSBwcm9jZXNzZWQgbmVhcmVzdC10by10aGUtdGFyZ2V0LWVkZ2UgZmlyc3Qgc28gdGhhdCBwaWxlcwoKICAgIGZvcm0gZGV0ZXJtaW5pc3RpY2FsbHk7IHBhc3NlcyByZXBlYXQgdW50aWwgbm90aGluZyBtb3Zlcy4KCiAgICAiIiIKCiAgICBjb21wcyA9IGNvbXBvbmVudHMoZywgY29ubmVjdGl2aXR5LCBiZykKCiAgICBpZiBub3QgY29tcHM6CgogICAgICAgIHJldHVybiBnCgogICAgaCwgdyA9IHNoYXBlKGcpCgogICAgaWYgZGlyZWN0aW9uID09ICJkb3duIjoKCiAgICAgICAgZHIsIGRjID0gMSwgMAoKICAgICAgICBjb21wcyA9IHNvcnRlZChjb21wcywga2V5PWxhbWJkYSBvOiAtby5iYm94WzJdKQoKICAgIGVsaWYgZGlyZWN0aW9uID09ICJ1cCI6CgogICAgICAgIGRyLCBkYyA9IC0xLCAwCgogICAgICAgIGNvbXBzID0gc29ydGVkKGNvbXBzLCBrZXk9bGFtYmRhIG86IG8uYmJveFswXSkKCiAgICBlbGlmIGRpcmVjdGlvbiA9PSAicmlnaHQiOgoKICAgICAgICBkciwgZGMgPSAwLCAxCgogICAgICAgIGNvbXBzID0gc29ydGVkKGNvbXBzLCBrZXk9bGFtYmRhIG86IC1vLmJib3hbM10pCgogICAgZWxzZToKCiAgICAgICAgZHIsIGRjID0gMCwgLTEKCiAgICAgICAgY29tcHMgPSBzb3J0ZWQoY29tcHMsIGtleT1sYW1iZGEgbzogby5iYm94WzFdKQoKICAgIGNlbGxzID0gW3NldChvLmNlbGxzKSBmb3IgbyBpbiBjb21wc10KCiAgICBtb3ZlZCA9IFRydWUKCiAgICB3aGlsZSBtb3ZlZDoKCiAgICAgICAgbW92ZWQgPSBGYWxzZQoKICAgICAgICBmb3IgaSBpbiByYW5nZShsZW4oY2VsbHMpKToKCiAgICAgICAgICAgIG90aGVycyA9IHNldCgpCgogICAgICAgICAgICBmb3IgaiBpbiByYW5nZShsZW4oY2VsbHMpKToKCiAgICAgICAgICAgICAgICBpZiBqICE9IGk6CgogICAgICAgICAgICAgICAgICAgIG90aGVycyB8PSBjZWxsc1tqXQoKICAgICAgICAgICAgd2hpbGUgX3NoaWZ0X29rKGNlbGxzW2ldLCBkciwgZGMsIGgsIHcsIG90aGVycyk6CgogICAgICAgICAgICAgICAgY2VsbHNbaV0gPSB7KHIgKyBkciwgYyArIGRjKSBmb3IgciwgYyBpbiBjZWxsc1tpXX0KCiAgICAgICAgICAgICAgICBvdGhlcnMgPSB7KHIgKyBkciwgYyArIGRjKSBmb3IgciwgYyBpbiBvdGhlcnN9CgogICAgICAgICAgICAgICAgbW92ZWQgPSBUcnVlCgogICAgYmdjID0gYmcgaWYgYmcgaXMgbm90IE5vbmUgZWxzZSAwCgogICAgb3V0ID0gW1tiZ2NdICogdyBmb3IgXyBpbiByYW5nZShoKV0KCiAgICBmb3IgbywgY3MgaW4gemlwKGNvbXBzLCBjZWxscyk6CgogICAgICAgIGZvciByLCBjIGluIGNzOgoKICAgICAgICAgICAgb3V0W3JdW2NdID0gby5jb2xvcgoKICAgIHJldHVybiB0dXBsZSh0dXBsZShyb3cpIGZvciByb3cgaW4gb3V0KQoKCgoKCmRlZiBleHRyYWN0X29iamVjdHMoZywgb3JpZW50YXRpb249ImgiLCBvcmRlcj0icmVhZGluZyIsIGdhcD0wLAoKICAgICAgICAgICAgICAgICAgICBiZz1Ob25lLCBjb25uZWN0aXZpdHk9NCk6CgogICAgIiIiQ29sbGVjdCBldmVyeSBjb21wb25lbnQgb250byBhIGZyZXNoIGNhbnZhcywgYXJyYW5nZWQgaW4gYSByb3cgb3IKCiAgICBjb2x1bW4gKHRvcC9sZWZ0IGFsaWduZWQpLCBzZXBhcmF0ZWQgYnkgYGdhcGAgYmFja2dyb3VuZCBjZWxscy4iIiIKCiAgICBjb21wcyA9IGNvbXBvbmVudHMoZywgY29ubmVjdGl2aXR5LCBiZykKCiAgICBpZiBub3QgY29tcHM6CgogICAgICAgIHJldHVybiBnCgogICAgaWYgYmcgaXMgTm9uZToKCiAgICAgICAgYmcgPSBjb21wc1swXS5iZwoKICAgIGlmIG9yZGVyID09ICJzaXplX2Rlc2MiOgoKICAgICAgICBjb21wcyA9IHNvcnRlZChjb21wcywga2V5PWxhbWJkYSBvOiAoLW8uc2l6ZSwgby5iYm94KSkKCiAgICBlbGlmIG9yZGVyID09ICJzaXplX2FzYyI6CgogICAgICAgIGNvbXBzID0gc29ydGVkKGNvbXBzLCBrZXk9bGFtYmRhIG86IChvLnNpemUsIG8uYmJveCkpCgogICAgZWxpZiBvcmRlciA9PSAiY29sb3IiOgoKICAgICAgICBjb21wcyA9IHNvcnRlZChjb21wcywga2V5PWxhbWJkYSBvOiAoby5jb2xvciwgby5iYm94KSkKCiAgICBzdWJzID0gW28uc3ViZ3JpZCgpIGZvciBvIGluIGNvbXBzXQoKICAgIGhlaWdodHMgPSBbaGVpZ2h0KHMpIGZvciBzIGluIHN1YnNdCgogICAgd2lkdGhzID0gW3dpZHRoKHMpIGZvciBzIGluIHN1YnNdCgogICAgaWYgb3JpZW50YXRpb24gPT0gImgiOgoKICAgICAgICBvaCA9IG1heChoZWlnaHRzKQoKICAgICAgICBvdyA9IHN1bSh3aWR0aHMpICsgZ2FwICogKGxlbihzdWJzKSAtIDEpCgogICAgICAgIG91dCA9IFtbYmddICogb3cgZm9yIF8gaW4gcmFuZ2Uob2gpXQoKICAgICAgICBjID0gMAoKICAgICAgICBmb3IgcywgaGgsIHd3IGluIHppcChzdWJzLCBoZWlnaHRzLCB3aWR0aHMpOgoKICAgICAgICAgICAgZm9yIHIgaW4gcmFuZ2UoaGgpOgoKICAgICAgICAgICAgICAgIGZvciBjYyBpbiByYW5nZSh3dyk6CgogICAgICAgICAgICAgICAgICAgIG91dFtyXVtjICsgY2NdID0gc1tyXVtjY10KCiAgICAgICAgICAgIGMgKz0gd3cgKyBnYXAKCiAgICBlbHNlOgoKICAgICAgICBvaCA9IHN1bShoZWlnaHRzKSArIGdhcCAqIChsZW4oc3VicykgLSAxKQoKICAgICAgICBvdyA9IG1heCh3aWR0aHMpCgogICAgICAgIG91dCA9IFtbYmddICogb3cgZm9yIF8gaW4gcmFuZ2Uob2gpXQoKICAgICAgICByID0gMAoKICAgICAgICBmb3IgcywgaGgsIHd3IGluIHppcChzdWJzLCBoZWlnaHRzLCB3aWR0aHMpOgoKICAgICAgICAgICAgZm9yIHJyIGluIHJhbmdlKGhoKToKCiAgICAgICAgICAgICAgICBmb3IgY2MgaW4gcmFuZ2Uod3cpOgoKICAgICAgICAgICAgICAgICAgICBvdXRbciArIHJyXVtjY10gPSBzW3JyXVtjY10KCiAgICAgICAgICAgIHIgKz0gaGggKyBnYXAKCiAgICByZXR1cm4gdHVwbGUodHVwbGUocm93KSBmb3Igcm93IGluIG91dCk=").decode("utf-8"))


In [ ]:
import base64 as _b, os as _o
_PATH = "/kaggle/working/arcagi2/patterns.py"
_o.makedirs(_o.path.dirname(_PATH), exist_ok=True)
with open(_PATH, "w", encoding="utf-8") as _f:
    _f.write(_b.b64decode("IiIiUGF0dGVybi1jb21wbGV0aW9uIHRyYW5zZm9ybXM6IHJlcGFpciBjZWxscyB0aGF0IGJyZWFrIGxvY2FsIHJlZ3VsYXJpdHkuCgoKCkFSQy1BR0ktMiBjb250YWlucyB0YXNrcyB3aG9zZSBvdXRwdXQgaXMgdGhlIGlucHV0IHdpdGggYSBmZXcgYW5vbWFsb3VzCgpjZWxscyByZXN0b3JlZCB0byB0aGUgbG9jYWwgcGF0dGVybi4gVGhlc2UgdHJhbnNmb3JtcyBkZXJpdmUgdGhlIGV4cGVjdGVkCgp2YWx1ZSBmcm9tIGxvY2FsIGV2aWRlbmNlIG9ubHkgKHBlcmlvZGljaXR5IGFsb25nIGEgcm93L2NvbHVtbiwgb3IgdGhlCgpuZWlnaGJvcmhvb2QgbWFqb3JpdHkpIGFuZCBjaGFuZ2UgYSBjZWxsIG9ubHkgd2hlbiB0aGF0IGV2aWRlbmNlIGlzCgp1bmFtYmlndW91cy4gV2hldGhlciB0aGUgcmVwYWlyIGlzIGNvcnJlY3QgaXMgZGVjaWRlZCBieSBleGFjdCBjb25zaXN0ZW5jeQoKd2l0aCB0aGUgZGVtb25zdHJhdGlvbnMsIG5ldmVyIGJ5IHRoZSB0cmFuc2Zvcm0gaXRzZWxmLgoKCgpQZXJpb2QgZGV0ZWN0aW9uIGlzIHRoZSBzdWJ0bGUgcGFydC4gQSBwZXJpb2RpYyBpbnRlcmlvciBmcmFtZWQgYnkgY29uc3RhbnQKCmJvcmRlcnMgKGEgdmVyeSBjb21tb24gQVJDIGxheW91dCkgZGVmZWF0cyBuYWl2ZSBkZXRlY3RvcnM6IHZpb2xhdGlvbgoKY291bnRzIHBpY2sgdGhlIGZyYW1lJ3MgcGVyaW9kLCBhbmQgc21hbGxlc3QtYWdyZWVpbmctcGVyaW9kIHBpY2tzIHVwCgpjb2luY2lkZW50YWwgZGlzdGFuY2UtMSBhZ3JlZW1lbnQgaW5zaWRlIGFsdGVybmF0aW5nIHJ1bnMuIFRoZSBkZXRlY3RvcgoKdXNlZCBoZXJlIHNjb3JlcyBlYWNoIGNhbmRpZGF0ZSBwZXJpb2QgYnkgaG93IG1hbnkgY2VsbHMgYWdyZWUgd2l0aCB0aGVpcgoKcmVzaWR1ZS1jbGFzcyBtYWpvcml0eSwgbWVhc3VyZWQgb24gdGhlIGxpbmUgd2l0aCBpdHMgZmlyc3QgYW5kIGxhc3QgY2VsbAoKZHJvcHBlZCAoZnJhbWVzIGxpdmUgYXQgdGhlIGVkZ2VzKSwgYW5kIGJyZWFrcyB0aWVzIHRvd2FyZCB0aGUgc21hbGxlcgoKcGVyaW9kLiBBIGNlbGwgaXMgdGhlbiByZXBhaXJlZCBvbmx5IHdoZW4gdGhlIGV2aWRlbmNlIGlzIHVuYW1iaWd1b3VzIOKAlAoKYm90aCBzYW1lLXBlcmlvZCBuZWlnaGJvdXJzIGFncmVlLCBvciB0aGUgY2VsbCBpcyB0aGUgbG9uZSBkaXNzZW50ZXIgaW4gYQoKc3RyaWN0bHktbWFqb3JpdHkgY2xhc3Mgd2l0aCBzYW1lLXBlcmlvZCBuZWlnaGJvdXJzIG9uIGJvdGggc2lkZXMuCgoiIiIKCgoKZnJvbSBjb2xsZWN0aW9ucyBpbXBvcnQgQ291bnRlcgoKCgpmcm9tIC4gaW1wb3J0IGdyaWRzCgpmcm9tIC5ncmlkcyBpbXBvcnQgR3JpZCwgc2hhcGUsIGNlbGwKCgoKCgpkZWYgX2NsYXNzX21ham9yaXR5X3RvdGFsKGxpbmUsIHApOgoKICAgICIiIkNlbGxzIGFncmVlaW5nIHdpdGggdGhlaXIgcmVzaWR1ZS1jbGFzcyBtYWpvcml0eSBmb3IgcGVyaW9kIHAuIiIiCgogICAgdG90YWwgPSAwCgogICAgZm9yIHIgaW4gcmFuZ2UocCk6CgogICAgICAgIHZhbHMgPSBbbGluZVtjXSBmb3IgYyBpbiByYW5nZShyLCBsZW4obGluZSksIHApXQoKICAgICAgICBpZiBub3QgdmFsczoKCiAgICAgICAgICAgIGNvbnRpbnVlCgogICAgICAgIGNvdW50cyA9IENvdW50ZXIodmFscykKCiAgICAgICAgdG90YWwgKz0gbWF4KGNvdW50cy52YWx1ZXMoKSkKCiAgICByZXR1cm4gdG90YWwKCgoKCgpkZWYgX2RvbWluYW50X3BlcmlvZChsaW5lLCBtYXhfcGVyaW9kPTQpOgoKICAgICIiIlRoZSBsaW5lJ3MgcGVyaW9kOiBtb3N0IGNsYXNzLW1ham9yaXR5IGFncmVlbWVudCwgc21hbGxlc3Qgb24gdGllcy4KCgoKICAgIFRoZSBmaXJzdCBhbmQgbGFzdCBjZWxscyBhcmUgZHJvcHBlZCBiZWZvcmUgc2NvcmluZyBiZWNhdXNlIGZyYW1lcyBhbmQKCiAgICBib3JkZXJzIGxpdmUgYXQgbGluZSBlZGdlcyBhbmQgd291bGQgb3RoZXJ3aXNlIGRyYWcgdGhlIGVzdGltYXRlIHRvd2FyZAoKICAgIHRoZSBmcmFtZSdzIHBlcmlvZC4KCiAgICAiIiIKCiAgICBjb3JlID0gbGluZVsxOi0xXSBpZiBsZW4obGluZSkgPiAyIGVsc2UgbGluZQoKICAgIG4gPSBsZW4oY29yZSkKCiAgICBiZXN0X3AsIGJlc3RfcyA9IDEsIC0xCgogICAgZm9yIHAgaW4gcmFuZ2UoMSwgbWluKG1heF9wZXJpb2QsIG4gLSAxKSArIDEpOgoKICAgICAgICBzID0gX2NsYXNzX21ham9yaXR5X3RvdGFsKGNvcmUsIHApCgogICAgICAgIGlmIHMgPiBiZXN0X3M6CgogICAgICAgICAgICBiZXN0X3AsIGJlc3RfcyA9IHAsIHMKCiAgICByZXR1cm4gYmVzdF9wCgoKCgoKZGVmIF9yZXBhaXJfbGluZShsaW5lLCBtYXhfcGVyaW9kPTQpOgoKICAgICIiIlJlcGFpciBjZWxscyB0aGF0IGNvbnRyYWRpY3QgdGhlIGxpbmUncyBkb21pbmFudCBwZXJpb2QuCgoKCiAgICBQZXJpb2QgcCBpcyBkZXRlY3RlZCBvbiB0aGUgbGluZSdzIGNvcmUgKGZpcnN0L2xhc3QgY2VsbCBkcm9wcGVkKSBieQoKICAgIHJlc2lkdWUtY2xhc3MgbWFqb3JpdHkgYWdyZWVtZW50LiBUd28gdW5hbWJpZ3VvdXMtZXZpZGVuY2UgcnVsZXMgdGhlbgoKICAgIHJ1biBvbiB0aGUgZnVsbCBsaW5lLCBib3RoIGdhdGVkIHNvIGEgZnJhbWUgY2VsbCBjYW4gbmV2ZXIgYmUgdG91Y2hlZDoKCgoKICAgIDEuIGJvdGgtbmVpZ2hib3VyczogdGhlIGNlbGxzIGF0IGRpc3RhbmNlIHAgb24gZWl0aGVyIHNpZGUgZXhpc3QgYW5kCgogICAgICAgYWdyZWUgd2l0aCBlYWNoIG90aGVyIGJ1dCBub3Qgd2l0aCB0aGUgY2VsbCwgYW5kIHRoYXQgYWdyZWVkIHZhbHVlCgogICAgICAgaXMgbm90IGNvbnRyYWRpY3RlZCBieSBhIHN0cmljdCBjbGFzcyBtYWpvcml0eSAtPiByZXN0b3JlIGl0LgoKICAgIDIuIGxvbmUtZGlzc2VudGVyOiB0aGUgY2VsbCBzaXRzIGF0IGxlYXN0IHR3byBjZWxscyBpbiBmcm9tIGVhY2ggbGluZQoKICAgICAgIGVkZ2UgKGZyYW1lIHRlcnJpdG9yeSBpcyBjbG9zZXIpLCBpcyB0aGUgb25seSBvY2N1cnJlbmNlIG9mIGl0cwoKICAgICAgIHZhbHVlIGluc2lkZSBpdHMgcmVzaWR1ZSBjbGFzcywgYW5kIHRoZSBjbGFzcyBoYXMgYSBzdHJpY3QgbWFqb3JpdHkKCiAgICAgICAtPiB0YWtlIHRoZSBtYWpvcml0eS4KCgoKICAgIEFueXRoaW5nIGFtYmlndW91cyAoY29uZmxpY3RpbmcgZXZpZGVuY2UsIG5vbi1zdHJpY3QgY2xhc3NlcywgY2VsbHMKCiAgICBodWdnaW5nIHRoZSBsaW5lIGVkZ2VzKSBpcyBsZWZ0IHVudG91Y2hlZC4gT3Zlci1yZXBhaXJpbmcgYSBmcmFtZSBjZWxsCgogICAgd291bGQgbWFrZSB0aGUgd2hvbGUgdHJhbnNmb3JtIGluY29uc2lzdGVudCB3aXRoIHRoZSBkZW1vbnN0cmF0aW9ucwoKICAgIGFuZCBnZXQgaXQgcmVqZWN0ZWQ7IHVuZGVyLXJlcGFpcmluZyBvbmx5IGNvc3RzIHRoYXQgb25lIHRhc2suCgogICAgIiIiCgogICAgcCA9IF9kb21pbmFudF9wZXJpb2QobGluZSwgbWF4X3BlcmlvZCkKCiAgICBuID0gbGVuKGxpbmUpCgogICAgY29yZSA9IGxpbmVbMTotMV0gaWYgbiA+IDIgZWxzZSBsaW5lCgogICAgIyBjbGFzcyBzdGF0aXN0aWNzIGNvbWUgZnJvbSB0aGUgY29yZSAoZnJhbWVzIGxpdmUgYXQgdGhlIGVkZ2VzKSwgYnV0IGEKCiAgICAjIGNvcmUgb2YgMS0yIGNlbGxzIGNhcnJpZXMgbm8gY2xhc3Mgc2lnbmFsIGF0IGFsbCAtPiBmYWxsIGJhY2sgdG8gdGhlCgogICAgIyBmdWxsIGxpbmUgZm9yIHNob3J0IGxpbmVzCgogICAgc3JjLCBvZmYgPSAoY29yZSwgMSkgaWYgbGVuKGNvcmUpID49IDMgZWxzZSAobGluZSwgMCkKCiAgICBvdXQgPSBsaXN0KGxpbmUpCgogICAgZm9yIGMgaW4gcmFuZ2Uobik6CgogICAgICAgIGN1ciA9IGxpbmVbY10KCiAgICAgICAgY2kgPSBjIC0gb2ZmCgogICAgICAgIGNscyA9IFtzcmNbaV0gZm9yIGkgaW4gcmFuZ2UoY2kgJSBwLCBsZW4oc3JjKSwgcCldIGlmIHNyYyBlbHNlIFtdCgogICAgICAgIGNvdW50cyA9IENvdW50ZXIoY2xzKQoKICAgICAgICBtYWpvcml0eSwgc3RyaWN0ID0gTm9uZSwgRmFsc2UKCiAgICAgICAgaWYgY291bnRzOgoKICAgICAgICAgICAgdG9wX24gPSBtYXgoY291bnRzLnZhbHVlcygpKQoKICAgICAgICAgICAgaWYgdG9wX24gKiAyID4gbGVuKGNscyk6CgogICAgICAgICAgICAgICAgbWFqb3JpdHkgPSBtYXgoY291bnRzLml0ZW1zKCksCgogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAga2V5PWxhbWJkYSBrdjogKGt2WzFdLCAta3ZbMF0pKVswXQoKICAgICAgICAgICAgICAgIHN0cmljdCA9IFRydWUKCiAgICAgICAgYSA9IGxpbmVbYyAtIHBdIGlmIGMgLSBwID49IDAgZWxzZSBOb25lCgogICAgICAgIGIgPSBsaW5lW2MgKyBwXSBpZiBjICsgcCA8IG4gZWxzZSBOb25lCgogICAgICAgIGlmIChhIGlzIG5vdCBOb25lIGFuZCBiIGlzIG5vdCBOb25lIGFuZCBhID09IGIgYW5kIGEgIT0gY3VyCgogICAgICAgICAgICAgICAgYW5kIChub3Qgc3RyaWN0IG9yIGEgPT0gbWFqb3JpdHkpKToKCiAgICAgICAgICAgIG91dFtjXSA9IGEKCiAgICAgICAgICAgIGNvbnRpbnVlCgogICAgICAgIGlmIChzdHJpY3QgYW5kIG1ham9yaXR5ICE9IGN1ciBhbmQgY291bnRzLmdldChjdXIsIDApID09IDEKCiAgICAgICAgICAgICAgICBhbmQgMiA8PSBjIDw9IG4gLSAzKToKCiAgICAgICAgICAgIG91dFtjXSA9IG1ham9yaXR5CgogICAgcmV0dXJuIG91dAoKCgoKCmRlZiBmaXhfcGVyaW9kKGcsIGF4aXM9ImJvdGgiLCBtYXhfcGVyaW9kPTQpOgoKICAgICIiIlJlc3RvcmUgY2VsbHMgdGhhdCBjb250cmFkaWN0IHRoZSBsb2NhbCBwZXJpb2RpYyBwYXR0ZXJuLiIiIgoKICAgIGgsIHcgPSBzaGFwZShnKQoKICAgIHJvd3MgPSBbbGlzdChyKSBmb3IgciBpbiBnXQoKICAgIGlmIGF4aXMgaW4gKCJyb3ciLCAiYm90aCIpOgoKICAgICAgICBmb3IgciBpbiByYW5nZShoKToKCiAgICAgICAgICAgIHJvd3Nbcl0gPSBfcmVwYWlyX2xpbmUocm93c1tyXSwgbWF4X3BlcmlvZCkKCiAgICBpZiBheGlzIGluICgiY29sIiwgImJvdGgiKToKCiAgICAgICAgY29scyA9IFtbcm93c1tyXVtjXSBmb3IgciBpbiByYW5nZShoKV0gZm9yIGMgaW4gcmFuZ2UodyldCgogICAgICAgIGNvbHMgPSBbX3JlcGFpcl9saW5lKGNvbCwgbWF4X3BlcmlvZCkgZm9yIGNvbCBpbiBjb2xzXQoKICAgICAgICBmb3IgYyBpbiByYW5nZSh3KToKCiAgICAgICAgICAgIGZvciByIGluIHJhbmdlKGgpOgoKICAgICAgICAgICAgICAgIHJvd3Nbcl1bY10gPSBjb2xzW2NdW3JdCgogICAgcmV0dXJuIHR1cGxlKHR1cGxlKHIpIGZvciByIGluIHJvd3MpCgoKCgoKZGVmIGZpeF9tYWpvcml0eShnKToKCiAgICAiIiJSZXBhaW50IGNlbGxzIHRoYXQgZGlzYWdyZWUgd2l0aCBBTEwgZWlnaHQgbmVpZ2hib3Vycy4KCgoKICAgIFRoZSBuZWlnaGJvcmhvb2QgbWFqb3JpdHkgaXMgdGhlIGV4cGVjdGF0aW9uOyBhIGNlbGwgaXMgb25seSByZXBhaXJlZAoKICAgIHdoZW4gaXQgbWF0Y2hlcyBub25lIG9mIGl0cyBuZWlnaGJvdXJzIChhIGdlbnVpbmUgb3V0bGllciksIHNvCgogICAgc3RydWN0dXJlZCByZWdpb25zIGFuZCBib3VuZGFyaWVzIHN1cnZpdmUuCgogICAgIiIiCgogICAgaCwgdyA9IHNoYXBlKGcpCgogICAgb3V0ID0gW2xpc3Qocm93KSBmb3Igcm93IGluIGddCgogICAgZm9yIHIgaW4gcmFuZ2UoaCk6CgogICAgICAgIGZvciBjIGluIHJhbmdlKHcpOgoKICAgICAgICAgICAgdiA9IGdbcl1bY10KCiAgICAgICAgICAgIG5laWdoYm9ycyA9IFtdCgogICAgICAgICAgICBmb3IgZHIgaW4gKC0xLCAwLCAxKToKCiAgICAgICAgICAgICAgICBmb3IgZGMgaW4gKC0xLCAwLCAxKToKCiAgICAgICAgICAgICAgICAgICAgaWYgZHIgPT0gMCBhbmQgZGMgPT0gMDoKCiAgICAgICAgICAgICAgICAgICAgICAgIGNvbnRpbnVlCgogICAgICAgICAgICAgICAgICAgIG52ID0gY2VsbChnLCByICsgZHIsIGMgKyBkYykKCiAgICAgICAgICAgICAgICAgICAgaWYgbnYgaXMgbm90IE5vbmU6CgogICAgICAgICAgICAgICAgICAgICAgICBuZWlnaGJvcnMuYXBwZW5kKG52KQoKICAgICAgICAgICAgaWYgbm90IG5laWdoYm9ycyBvciB2IGluIG5laWdoYm9yczoKCiAgICAgICAgICAgICAgICBjb250aW51ZQoKICAgICAgICAgICAgY291bnRzID0gQ291bnRlcihuZWlnaGJvcnMpCgogICAgICAgICAgICBiZXN0ID0gbWF4KGNvdW50cy5pdGVtcygpLCBrZXk9bGFtYmRhIGt2OiAoa3ZbMV0sIC1rdlswXSkpWzBdCgogICAgICAgICAgICBvdXRbcl1bY10gPSBiZXN0CgogICAgcmV0dXJuIHR1cGxlKHR1cGxlKHJvdykgZm9yIHJvdyBpbiBvdXQpCgoKCgoKUEFUVEVSTl9NT0RFUyA9ICgicm93IiwgImNvbCIsICJib3RoIiwgIm1ham9yaXR5IikKCgoKCgpkZWYgYXBwbHlfcGF0dGVybihnLCBtb2RlKToKCiAgICBpZiBtb2RlID09ICJtYWpvcml0eSI6CgogICAgICAgIHJldHVybiBmaXhfbWFqb3JpdHkoZykKCiAgICByZXR1cm4gZml4X3BlcmlvZChnLCBheGlzPW1vZGUp").decode("utf-8"))


In [ ]:
import base64 as _b, os as _o
_PATH = "/kaggle/working/arcagi2/dsl.py"
_o.makedirs(_o.path.dirname(_PATH), exist_ok=True)
with open(_PATH, "w", encoding="utf-8") as _f:
    _f.write(_b.b64decode("IiIiVGhlIHRyYW5zZm9ybSBEU0w6IGEgc21hbGwsIGNsb3NlZCBsaWJyYXJ5IG9mIGdyaWQgdHJhbnNmb3JtYXRpb25zLgoKCgpFdmVyeSB0cmFuc2Zvcm0gaXMgYSBmcm96ZW4gcmVjb3JkIChmYW1pbHksIGFyZ3MpIHdpdGggYSBwdXJlIGFwcGx5KCkuIFRoZQoKc29sdmVyIHNlYXJjaGVzIGNvbXBvc2l0aW9ucyBvZiB0aGVzZSB0cmFuc2Zvcm1zOyBrZWVwaW5nIHRoZSBsaWJyYXJ5IGNsb3NlZAoKYW5kIGV4cGxpY2l0IGlzIHdoYXQgbWFrZXMgdGhlIHNlYXJjaCBleGhhdXN0aXZlIHJhdGhlciB0aGFuIGhldXJpc3RpYy4KCiIiIgoKCgpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MKCmZyb20gaXRlcnRvb2xzIGltcG9ydCBwcm9kdWN0CgoKCmZyb20gLiBpbXBvcnQgZ3JpZHMKCmZyb20gLiBpbXBvcnQgb2JqZWN0cyBhcyBvYmptb2QKCmZyb20gLiBpbXBvcnQgcGF0dGVybnMgYXMgcGF0bW9kCgpmcm9tIC5ncmlkcyBpbXBvcnQgR3JpZAoKCgoKCkBkYXRhY2xhc3MoZnJvemVuPVRydWUpCgpjbGFzcyBUOgoKICAgIGZhbWlseTogc3RyCgogICAgYXJnczogdHVwbGUKCgoKICAgIGRlZiBhcHBseShzZWxmLCBnKToKCiAgICAgICAgcmV0dXJuIF9BUFBMWVtzZWxmLmZhbWlseV0oZywgKnNlbGYuYXJncykKCgoKICAgIGRlZiBzaWduYXR1cmUoc2VsZik6CgogICAgICAgIHJldHVybiAoc2VsZi5mYW1pbHksIHNlbGYuYXJncykKCgoKICAgIGRlZiBfX3JlcHJfXyhzZWxmKToKCiAgICAgICAgaWYgc2VsZi5hcmdzOgoKICAgICAgICAgICAgcmV0dXJuICIlcyVzIiAlIChzZWxmLmZhbWlseSwgc2VsZi5hcmdzKQoKICAgICAgICByZXR1cm4gc2VsZi5mYW1pbHkKCgoKCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gZmFtaWx5IGFwcGxpZXJzCgoKCmRlZiBfaWRlbnRpdHkoZyk6CgogICAgcmV0dXJuIGcKCgoKCgpkZWYgX3JlY29sb3IoZywgKm1hcHBpbmcpOgoKICAgIHJldHVybiBncmlkcy5yZWNvbG9yKGcsIGRpY3QobWFwcGluZykpCgoKCgoKZGVmIF9zd2FwKGcsIGEsIGIpOgoKICAgIHJldHVybiBncmlkcy5zd2FwX2NvbG9ycyhnLCBhLCBiKQoKCgoKCmRlZiBfa2VlcChnLCAqY29sb3JzKToKCiAgICByZXR1cm4gZ3JpZHMua2VlcF9jb2xvcnMoZywgY29sb3JzKQoKCgoKCmRlZiBfY3JvcChnKToKCiAgICByZXR1cm4gZ3JpZHMuY3JvcF9iYm94KGcpCgoKCgoKZGVmIF9jcm9wX2IoZywgYmcpOgoKICAgIHJldHVybiBncmlkcy5jcm9wX2Jib3goZywgYmc9YmcpCgoKCgoKZGVmIF9wYWQoZywgaCwgdywgYW5jaG9yLCBiZyk6CgogICAgcmV0dXJuIGdyaWRzLnBhZF90byhnLCBoLCB3LCBiZz1iZywgYW5jaG9yPWFuY2hvcikKCgoKCgpkZWYgX3VwKGcsIGZoLCBmdyk6CgogICAgcmV0dXJuIGdyaWRzLnVwc2NhbGUoZywgZmgsIGZ3KQoKCgoKCmRlZiBfZG93bihnLCBmaCwgZncsIG1vZGUpOgoKICAgIHJldHVybiBncmlkcy5kb3duc2NhbGUoZywgZmgsIGZ3LCBtb2RlKQoKCgoKCmRlZiBfcm90OTAoZyk6CgogICAgcmV0dXJuIGdyaWRzLnJvdGF0ZTkwKGcpCgoKCgoKZGVmIF9yb3QxODAoZyk6CgogICAgcmV0dXJuIGdyaWRzLnJvdGF0ZTE4MChnKQoKCgoKCmRlZiBfcm90MjcwKGcpOgoKICAgIHJldHVybiBncmlkcy5yb3RhdGUyNzAoZykKCgoKCgpkZWYgX2ZsaXBfaChnKToKCiAgICByZXR1cm4gZ3JpZHMuZmxpcF9oKGcpCgoKCgoKZGVmIF9mbGlwX3YoZyk6CgogICAgcmV0dXJuIGdyaWRzLmZsaXBfdihnKQoKCgoKCmRlZiBfdHJhbnNwb3NlKGcpOgoKICAgIHJldHVybiBncmlkcy50cmFuc3Bvc2UoZykKCgoKCgpkZWYgX2FudGkoZyk6CgogICAgcmV0dXJuIGdyaWRzLmFudGlfdHJhbnNwb3NlKGcpCgoKCgoKZGVmIF90cmFuc2xhdGUoZywgZGgsIGR3LCBiZyk6CgogICAgcmV0dXJuIGdyaWRzLnRyYW5zbGF0ZShnLCBkaCwgZHcsIGJnPWJnKQoKCgoKCmRlZiBfdGlsZShnLCBuaCwgbncpOgoKICAgIHJldHVybiBncmlkcy50aWxlKGcsIG5oLCBudykKCgoKCgpkZWYgX2hvbGVzKGcsIGhvbGUsIGNvbG9yKToKCiAgICByZXR1cm4gZ3JpZHMuZmlsbF9ob2xlcyhnLCBob2xlPWhvbGUsIGNvbG9yPWNvbG9yKQoKCgoKCmRlZiBfc3ltX2goZyk6CgogICAgcmV0dXJuIGdyaWRzLmNvbXBsZXRlX2hfc3ltbWV0cnkoZykKCgoKCgpkZWYgX3N5bV92KGcpOgoKICAgIHJldHVybiBncmlkcy5jb21wbGV0ZV92X3N5bW1ldHJ5KGcpCgoKCgoKZGVmIF9ncmF2aXR5KGcsIGRpcmVjdGlvbik6CgogICAgcmV0dXJuIGdyaWRzLmdyYXZpdHkoZywgZGlyZWN0aW9uKQoKCgoKCmRlZiBfYm9yZGVyKGcsIGNvbG9yLCB0aGlja25lc3MpOgoKICAgIHJldHVybiBncmlkcy5ib3JkZXIoZywgY29sb3IsIHRoaWNrbmVzcykKCgoKCgpkZWYgX291dGxpbmUoZywgY29sb3IpOgoKICAgIHJldHVybiBncmlkcy5vdXRsaW5lX25vbmJnKGcsIGNvbG9yKQoKCgoKCmRlZiBfa2VlcF9sYXJnZXN0KGcsIGNvbm5lY3Rpdml0eSk6CgogICAgY29tcHMgPSBvYmptb2QuY29tcG9uZW50cyhnLCBjb25uZWN0aXZpdHkpCgogICAgYmlnID0gb2JqbW9kLmxhcmdlc3QoY29tcHMpCgogICAgaWYgYmlnIGlzIE5vbmU6CgogICAgICAgIHJldHVybiBnCgogICAgcmV0dXJuIG9iam1vZC5wYWludChncmlkcy5ibGFuayhncmlkcy5oZWlnaHQoZyksIGdyaWRzLndpZHRoKGcpKSwKCiAgICAgICAgICAgICAgICAgICAgICAgIFtiaWddKQoKCgoKCmRlZiBfZGVsZXRlX2NvbG9yKGcsIGNvbG9yKToKCiAgICBiZyA9IGdyaWRzLmJhY2tncm91bmRfY29sb3IoZykKCiAgICByZXR1cm4gZ3JpZHMucmVwbGFjZV9jb2xvcihnLCBjb2xvciwgYmcpCgoKCgoKZGVmIF9maWxsX2Jib3goZywgc3JjLCBkc3QpOgoKICAgICIiIkZpbGwgdGhlIGJvdW5kaW5nIGJveCBvZiBldmVyeSBvYmplY3Qgb2YgY29sb3IgYHNyY2Agd2l0aCBgZHN0YC4iIiIKCiAgICBiZyA9IGdyaWRzLmJhY2tncm91bmRfY29sb3IoZykKCiAgICBjb21wcyA9IG9iam1vZC5jb21wb25lbnRzKGcsIDQsIGJnKQoKICAgIGgsIHcgPSBncmlkcy5zaGFwZShnKQoKICAgIG91dCA9IFtsaXN0KHJvdykgZm9yIHJvdyBpbiBnXQoKICAgIGZvciBvIGluIGNvbXBzOgoKICAgICAgICBpZiBvLmNvbG9yICE9IHNyYzoKCiAgICAgICAgICAgIGNvbnRpbnVlCgogICAgICAgIHIwLCBjMCwgcjEsIGMxID0gby5iYm94CgogICAgICAgIGZvciByIGluIHJhbmdlKHIwLCByMSArIDEpOgoKICAgICAgICAgICAgZm9yIGMgaW4gcmFuZ2UoYzAsIGMxICsgMSk6CgogICAgICAgICAgICAgICAgb3V0W3JdW2NdID0gZHN0CgogICAgcmV0dXJuIHR1cGxlKHR1cGxlKHJvdykgZm9yIHJvdyBpbiBvdXQpCgoKCgoKZGVmIF9vYmpfZ3Jhdml0eShnLCBkaXJlY3Rpb24pOgoKICAgIHJldHVybiBvYmptb2Qub2JqZWN0X2dyYXZpdHkoZywgZGlyZWN0aW9uKQoKCgoKCmRlZiBfZXh0cmFjdChnLCBvcmllbnRhdGlvbiwgb3JkZXIsIGdhcCk6CgogICAgcmV0dXJuIG9iam1vZC5leHRyYWN0X29iamVjdHMoZywgb3JpZW50YXRpb24sIG9yZGVyLCBnYXApCgoKCgoKZGVmIF9wYXR0ZXJuKGcsIG1vZGUpOgoKICAgIHJldHVybiBwYXRtb2QuYXBwbHlfcGF0dGVybihnLCBtb2RlKQoKCgoKCmRlZiBfb2JqX3JlbWFwKGcsIGJnX291dCwgKm1hcHBpbmcpOgoKICAgICIiIlBlci1jb2xvciBvYmplY3QgdHJhbnNmb3JtOiBldmVyeSBjb21wb25lbnQgb2YgY29sb3IgYyBpcyBzaGlmdGVkIGJ5CgogICAgKGRyLCBkYykgYW5kIHJlY29sb3JlZCB0byBuYywgcGVyIHRoZSBpbmZlcnJlZCB0YWJsZS4gQ29tcG9uZW50cyB3aG9zZQoKICAgIGNvbG9yIGlzIGFic2VudCBmcm9tIHRoZSB0YWJsZSBzdGF5IGluIHBsYWNlOyB0aGUgY2FudmFzIGlzIHJlYnVpbHQgb24KCiAgICBiZ19vdXQgc28gYSBiYWNrZ3JvdW5kIGNoYW5nZSBpcyBleHByZXNzZWQgdG9vLiIiIgoKICAgIHRhYmxlID0ge21bMF06IChtWzFdLCBtWzJdLCBtWzNdKSBmb3IgbSBpbiBtYXBwaW5nfQoKICAgIGJnID0gZ3JpZHMuYmFja2dyb3VuZF9jb2xvcihnKQoKICAgIGNvbXBzID0gb2JqbW9kLmNvbXBvbmVudHMoZywgNCwgYmcpCgogICAgaCwgdyA9IGdyaWRzLnNoYXBlKGcpCgogICAgb3V0ID0gW1tiZ19vdXRdICogdyBmb3IgXyBpbiByYW5nZShoKV0KCiAgICBmb3IgbyBpbiBjb21wczoKCiAgICAgICAgZW50cnkgPSB0YWJsZS5nZXQoby5jb2xvcikKCiAgICAgICAgaWYgZW50cnkgaXMgTm9uZToKCiAgICAgICAgICAgIGNlbGxzLCBjb2xvciA9IG8uY2VsbHMsIG8uY29sb3IKCiAgICAgICAgZWxzZToKCiAgICAgICAgICAgIGRyLCBkYywgbmMgPSBlbnRyeQoKICAgICAgICAgICAgY2VsbHMgPSB7KHIgKyBkciwgYyArIGRjKSBmb3IgciwgYyBpbiBvLmNlbGxzfQoKICAgICAgICAgICAgY29sb3IgPSBuYwoKICAgICAgICBmb3IgciwgYyBpbiBjZWxsczoKCiAgICAgICAgICAgIGlmIDAgPD0gciA8IGggYW5kIDAgPD0gYyA8IHc6CgogICAgICAgICAgICAgICAgb3V0W3JdW2NdID0gY29sb3IKCiAgICByZXR1cm4gdHVwbGUodHVwbGUocm93KSBmb3Igcm93IGluIG91dCkKCgoKCgpfQVBQTFkgPSB7CgogICAgImlkZW50aXR5IjogX2lkZW50aXR5LAoKICAgICJyZWNvbG9yIjogX3JlY29sb3IsCgogICAgInN3YXAiOiBfc3dhcCwKCiAgICAia2VlcCI6IF9rZWVwLAoKICAgICJjcm9wIjogX2Nyb3AsCgogICAgImNyb3BfYiI6IF9jcm9wX2IsCgogICAgInBhZCI6IF9wYWQsCgogICAgInVwIjogX3VwLAoKICAgICJkb3duIjogX2Rvd24sCgogICAgInJvdDkwIjogX3JvdDkwLAoKICAgICJyb3QxODAiOiBfcm90MTgwLAoKICAgICJyb3QyNzAiOiBfcm90MjcwLAoKICAgICJmbGlwX2giOiBfZmxpcF9oLAoKICAgICJmbGlwX3YiOiBfZmxpcF92LAoKICAgICJ0cmFuc3Bvc2UiOiBfdHJhbnNwb3NlLAoKICAgICJhbnRpIjogX2FudGksCgogICAgInRyYW5zbGF0ZSI6IF90cmFuc2xhdGUsCgogICAgInRpbGUiOiBfdGlsZSwKCiAgICAiaG9sZXMiOiBfaG9sZXMsCgogICAgInN5bV9oIjogX3N5bV9oLAoKICAgICJzeW1fdiI6IF9zeW1fdiwKCiAgICAiZ3Jhdml0eSI6IF9ncmF2aXR5LAoKICAgICJib3JkZXIiOiBfYm9yZGVyLAoKICAgICJvdXRsaW5lIjogX291dGxpbmUsCgogICAgImtlZXBfbGFyZ2VzdCI6IF9rZWVwX2xhcmdlc3QsCgogICAgImRlbGV0ZV9jb2xvciI6IF9kZWxldGVfY29sb3IsCgogICAgImZpbGxfYmJveCI6IF9maWxsX2Jib3gsCgogICAgIm9ial9ncmF2aXR5IjogX29ial9ncmF2aXR5LAoKICAgICJleHRyYWN0IjogX2V4dHJhY3QsCgogICAgInBhdHRlcm4iOiBfcGF0dGVybiwKCiAgICAib2JqX3JlbWFwIjogX29ial9yZW1hcCwKCn0KCgoKQ09MT1JTID0gdHVwbGUocmFuZ2UoMTApKQoKQU5DSE9SUyA9ICgiY2VudGVyIiwgInRsIiwgInRyIiwgImJsIiwgImJyIikKCgoKCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gZW51bWVyYXRpb24KCgoKZGVmIGVudW1lcmF0ZWRfY2FuZGlkYXRlcygpOgoKICAgICIiIkV2ZXJ5IHRyYW5zZm9ybSBpbiB0aGUgbGlicmFyeSB3aXRoIGNvbmNyZXRlIChzZWFyY2hhYmxlKSBwYXJhbWV0ZXJzLgoKCgogICAgUmVjdXJzaW9ucy9tYXBwaW5ncyB0aGF0IG11c3QgYmUgKmluZmVycmVkKiBmcm9tIHRyYWluIHBhaXJzIChyZWNvbG9yCgogICAgbWFwcywgcGFkIHRhcmdldHMsIHRyYW5zbGF0aW9ucykgYXJlIGRlbGliZXJhdGVseSBleGNsdWRlZCBoZXJlIGFuZAoKICAgIHByb2R1Y2VkIGJ5IGluZmVyLnB5IGluc3RlYWQuCgogICAgIiIiCgogICAgb3V0ID0gW1QoImlkZW50aXR5IiwgKCkpXQoKCgogICAgZm9yIGEgaW4gQ09MT1JTOgoKICAgICAgICBmb3IgYiBpbiBDT0xPUlM6CgogICAgICAgICAgICBpZiBhICE9IGI6CgogICAgICAgICAgICAgICAgb3V0LmFwcGVuZChUKCJzd2FwIiwgKGEsIGIpKSkKCgoKICAgIGZvciBjIGluIENPTE9SUzoKCiAgICAgICAgb3V0LmFwcGVuZChUKCJrZWVwIiwgKGMsKSkpCgogICAgZm9yIGMgaW4gQ09MT1JTOgoKICAgICAgICBvdXQuYXBwZW5kKFQoImtlZXAiLCB0dXBsZSh4IGZvciB4IGluIENPTE9SUyBpZiB4ICE9IGMpKSkKCgoKICAgIG91dC5hcHBlbmQoVCgiY3JvcCIsICgpKSkKCgoKICAgIGZvciBmaCwgZncgaW4gcHJvZHVjdChyYW5nZSgyLCA2KSwgcmVwZWF0PTIpOgoKICAgICAgICBvdXQuYXBwZW5kKFQoInVwIiwgKGZoLCBmdykpKQoKCgogICAgZm9yIGZoLCBmdyBpbiBwcm9kdWN0KHJhbmdlKDIsIDUpLCByZXBlYXQ9Mik6CgogICAgICAgIGZvciBtb2RlIGluICgibWFqb3JpdHkiLCAiZmlyc3QiLCAidW5pcXVlIik6CgogICAgICAgICAgICBvdXQuYXBwZW5kKFQoImRvd24iLCAoZmgsIGZ3LCBtb2RlKSkpCgoKCiAgICBvdXQgKz0gW1QoInJvdDkwIiwgKCkpLCBUKCJyb3QxODAiLCAoKSksIFQoInJvdDI3MCIsICgpKSwKCiAgICAgICAgICAgIFQoImZsaXBfaCIsICgpKSwgVCgiZmxpcF92IiwgKCkpLAoKICAgICAgICAgICAgVCgidHJhbnNwb3NlIiwgKCkpLCBUKCJhbnRpIiwgKCkpXQoKCgogICAgZm9yIGRoIGluIHJhbmdlKC0zLCA0KToKCiAgICAgICAgZm9yIGR3IGluIHJhbmdlKC0zLCA0KToKCiAgICAgICAgICAgIGlmIGRoIG9yIGR3OgoKICAgICAgICAgICAgICAgIG91dC5hcHBlbmQoVCgidHJhbnNsYXRlIiwgKGRoLCBkdywgMCkpKQoKCgogICAgZm9yIG5oLCBudyBpbiBwcm9kdWN0KHJhbmdlKDEsIDQpLCByZXBlYXQ9Mik6CgogICAgICAgIGlmIG5oICogbncgPiAxOgoKICAgICAgICAgICAgb3V0LmFwcGVuZChUKCJ0aWxlIiwgKG5oLCBudykpKQoKCgogICAgb3V0ICs9IFtUKCJob2xlcyIsIChob2xlLCBjKSkgZm9yIGhvbGUgaW4gQ09MT1JTIGZvciBjIGluIENPTE9SUwoKICAgICAgICAgICAgaWYgaG9sZSAhPSBjXQoKICAgIG91dCArPSBbVCgic3ltX2giLCAoKSksIFQoInN5bV92IiwgKCkpXQoKCgogICAgZm9yIGQgaW4gKCJ1cCIsICJkb3duIiwgImxlZnQiLCAicmlnaHQiKToKCiAgICAgICAgb3V0LmFwcGVuZChUKCJncmF2aXR5IiwgKGQsKSkpCgoKCiAgICBmb3IgYyBpbiBDT0xPUlM6CgogICAgICAgIG91dC5hcHBlbmQoVCgiYm9yZGVyIiwgKGMsIDEpKSkKCiAgICAgICAgb3V0LmFwcGVuZChUKCJib3JkZXIiLCAoYywgMikpKQoKICAgICAgICBvdXQuYXBwZW5kKFQoIm91dGxpbmUiLCAoYywpKSkKCgoKICAgIG91dCArPSBbVCgia2VlcF9sYXJnZXN0IiwgKDQsKSksIFQoImtlZXBfbGFyZ2VzdCIsICg4LCkpXQoKCgogICAgZm9yIGMgaW4gQ09MT1JTOgoKICAgICAgICBpZiBjOgoKICAgICAgICAgICAgb3V0LmFwcGVuZChUKCJkZWxldGVfY29sb3IiLCAoYywpKSkKCgoKICAgIGZvciBzcmMgaW4gQ09MT1JTOgoKICAgICAgICBmb3IgZHN0IGluIENPTE9SUzoKCiAgICAgICAgICAgIGlmIHNyYyAhPSBkc3Q6CgogICAgICAgICAgICAgICAgb3V0LmFwcGVuZChUKCJmaWxsX2Jib3giLCAoc3JjLCBkc3QpKSkKCgoKICAgIGZvciBkIGluICgidXAiLCAiZG93biIsICJsZWZ0IiwgInJpZ2h0Iik6CgogICAgICAgIG91dC5hcHBlbmQoVCgib2JqX2dyYXZpdHkiLCAoZCwpKSkKCiAgICBmb3Igb3JpZW50YXRpb24gaW4gKCJoIiwgInYiKToKCiAgICAgICAgZm9yIG9yZGVyIGluICgicmVhZGluZyIsICJzaXplX2Rlc2MiLCAic2l6ZV9hc2MiLCAiY29sb3IiKToKCiAgICAgICAgICAgIGZvciBnYXAgaW4gKDAsIDEpOgoKICAgICAgICAgICAgICAgIG91dC5hcHBlbmQoVCgiZXh0cmFjdCIsIChvcmllbnRhdGlvbiwgb3JkZXIsIGdhcCkpKQoKICAgIGZvciBtb2RlIGluIHBhdG1vZC5QQVRURVJOX01PREVTOgoKICAgICAgICBvdXQuYXBwZW5kKFQoInBhdHRlcm4iLCAobW9kZSwpKSkKCgoKICAgIHJldHVybiBvdXQKCgoKCgpkZWYgc2Vjb25kX3N0YWdlX2NhbmRpZGF0ZXMoKToKCiAgICAiIiJBIGN1cmF0ZWQgc3Vic2V0IHVzZWQgYXMgdGhlICpzZWNvbmQqIHRyYW5zZm9ybSBpbiBjb21wb3NpdGlvbnMuCgoKCiAgICBTZWFyY2gtc3BhY2UgY29udHJvbDogYSBmdWxsIGNyb3NzIHByb2R1Y3Qgb2YgZXZlcnkgY2FuZGlkYXRlIHdpdGggZXZlcnkKCiAgICBjYW5kaWRhdGUgaXMgcXVhZHJhdGljOyByZXN0cmljdGluZyB0aGUgcmlnaHQtaGFuZCBzaWRlIHRvIHRoZSBmYW1pbGllcwoKICAgIHRoYXQgYWN0dWFsbHkgYXBwZWFyIGluIHNlY29uZCBwb3NpdGlvbiBrZWVwcyBkZXB0aC0yIHRyYWN0YWJsZSB3aXRob3V0CgogICAgbWVhc3VyYWJseSBsb3NpbmcgdGFza3MuCgogICAgIiIiCgogICAgb3V0ID0gW10KCiAgICBmb3IgYyBpbiBDT0xPUlM6CgogICAgICAgIG91dC5hcHBlbmQoVCgia2VlcCIsIChjLCkpKQoKICAgIGZvciBhIGluIENPTE9SUzoKCiAgICAgICAgZm9yIGIgaW4gQ09MT1JTOgoKICAgICAgICAgICAgaWYgYSAhPSBiOgoKICAgICAgICAgICAgICAgIG91dC5hcHBlbmQoVCgic3dhcCIsIChhLCBiKSkpCgogICAgb3V0LmFwcGVuZChUKCJjcm9wIiwgKCkpKQoKICAgIGZvciBmaCwgZncgaW4gcHJvZHVjdChyYW5nZSgyLCA2KSwgcmVwZWF0PTIpOgoKICAgICAgICBvdXQuYXBwZW5kKFQoInVwIiwgKGZoLCBmdykpKQoKICAgIGZvciBmaCwgZncgaW4gcHJvZHVjdChyYW5nZSgyLCA1KSwgcmVwZWF0PTIpOgoKICAgICAgICBvdXQuYXBwZW5kKFQoImRvd24iLCAoZmgsIGZ3LCAibWFqb3JpdHkiKSkpCgogICAgb3V0ICs9IFtUKCJob2xlcyIsIChob2xlLCBjKSkgZm9yIGhvbGUgaW4gQ09MT1JTIGZvciBjIGluIENPTE9SUwoKICAgICAgICAgICAgaWYgaG9sZSAhPSBjXQoKICAgIG91dCArPSBbVCgic3ltX2giLCAoKSksIFQoInN5bV92IiwgKCkpXQoKICAgIGZvciBkIGluICgidXAiLCAiZG93biIsICJsZWZ0IiwgInJpZ2h0Iik6CgogICAgICAgIG91dC5hcHBlbmQoVCgiZ3Jhdml0eSIsIChkLCkpKQoKICAgIGZvciBjIGluIENPTE9SUzoKCiAgICAgICAgb3V0LmFwcGVuZChUKCJib3JkZXIiLCAoYywgMSkpKQoKICAgICAgICBvdXQuYXBwZW5kKFQoIm91dGxpbmUiLCAoYywpKSkKCiAgICAgICAgaWYgYzoKCiAgICAgICAgICAgIG91dC5hcHBlbmQoVCgiZGVsZXRlX2NvbG9yIiwgKGMsKSkpCgogICAgZm9yIGQgaW4gKCJ1cCIsICJkb3duIiwgImxlZnQiLCAicmlnaHQiKToKCiAgICAgICAgb3V0LmFwcGVuZChUKCJvYmpfZ3Jhdml0eSIsIChkLCkpKQoKICAgIGZvciBvcmllbnRhdGlvbiBpbiAoImgiLCAidiIpOgoKICAgICAgICBmb3Igb3JkZXIgaW4gKCJyZWFkaW5nIiwgInNpemVfZGVzYyIsICJzaXplX2FzYyIsICJjb2xvciIpOgoKICAgICAgICAgICAgZm9yIGdhcCBpbiAoMCwgMSk6CgogICAgICAgICAgICAgICAgb3V0LmFwcGVuZChUKCJleHRyYWN0IiwgKG9yaWVudGF0aW9uLCBvcmRlciwgZ2FwKSkpCgogICAgZm9yIG1vZGUgaW4gcGF0bW9kLlBBVFRFUk5fTU9ERVM6CgogICAgICAgIG91dC5hcHBlbmQoVCgicGF0dGVybiIsIChtb2RlLCkpKQoKICAgIHJldHVybiBvdXQKCgoKCgpkZWYgYXBwbHlfcHJvZ3JhbShnLCBwcm9ncmFtKToKCiAgICBmb3IgdCBpbiBwcm9ncmFtOgoKICAgICAgICBnID0gdC5hcHBseShnKQoKICAgIHJldHVybiBnCgoKCgoKZGVmIHByb2dyYW1fY29zdChwcm9ncmFtKToKCiAgICAiIiJTaW1wbGljaXR5IHJhbmtpbmc6IGZld2VyIHRyYW5zZm9ybXMgZmlyc3QsIHRoZW4gZmV3ZXIgcGFyYW1ldGVycy4iIiIKCiAgICByZXR1cm4gKGxlbihwcm9ncmFtKSwgc3VtKGxlbih0LmFyZ3MpIGZvciB0IGluIHByb2dyYW0pKQ==").decode("utf-8"))


In [ ]:
import base64 as _b, os as _o
_PATH = "/kaggle/working/arcagi2/infer.py"
_o.makedirs(_o.path.dirname(_PATH), exist_ok=True)
with open(_PATH, "w", encoding="utf-8") as _f:
    _f.write(_b.b64decode("IiIiUGFyYW1ldGVyIGluZmVyZW5jZTogZGVyaXZlIHRyYW5zZm9ybSBwYXJhbWV0ZXJzIGZyb20gdGhlIGRlbW9uc3RyYXRpb25zLgoKCgpUaGUgQVJDLUFHSS0yIHNjb3JpbmcgZ2l2ZXMgY3JlZGl0IHdoZW4gKmVpdGhlciogb2YgdHdvIGF0dGVtcHRzIG1hdGNoZXMsIHNvCgpwcmVjaXNpb24gbWF0dGVycyBtb3JlIHRoYW4gcmVjYWxsOiBldmVyeSB0cmFuc2Zvcm0gaW5mZXJyZWQgaGVyZSBtdXN0CgpyZXByb2R1Y2UgQUxMIHRyYWluIHBhaXJzIGV4YWN0bHksIG9yIGl0IGlzIGRpc2NhcmRlZC4gQSB3cm9uZy1idXQtcGxhdXNpYmxlCgpndWVzcyB3b3VsZCB3YXN0ZSBvbmUgb2YgdGhlIHR3byBhdHRlbXB0cy4KCiIiIgoKCgppbXBvcnQgaXRlcnRvb2xzCgoKCmZyb20gLiBpbXBvcnQgZ3JpZHMKCmZyb20gLiBpbXBvcnQgb2JqZWN0cyBhcyBvYmptb2QKCmZyb20gLmRzbCBpbXBvcnQgVCwgQU5DSE9SUywgQ09MT1JTCgoKCgoKZGVmIF9jb25zaXN0ZW50KHQsIHBhaXJzKToKCiAgICAiIiJUcnVlIHdoZW4gdCByZXByb2R1Y2VzIGV2ZXJ5IChpbnB1dCwgb3V0cHV0KSB0cmFpbiBwYWlyIGV4YWN0bHkuIiIiCgogICAgZm9yIGdpLCBnbyBpbiBwYWlyczoKCiAgICAgICAgaWYgdC5hcHBseShnaSkgIT0gZ286CgogICAgICAgICAgICByZXR1cm4gRmFsc2UKCiAgICByZXR1cm4gVHJ1ZQoKCgoKCmRlZiBpbmZlcl9yZWNvbG9yX21hcChwYWlycyk6CgogICAgIiIiSWYgZXZlcnkgcGFpciBpcyBzaGFwZS1wcmVzZXJ2aW5nLCBkZXJpdmUgdGhlIGdsb2JhbCBjb2xvciBtYXAuCgoKCiAgICBPbmx5IGVtaXR0ZWQgd2hlbiB0aGUgY2VsbC13aXNlIG1hcCBpcyBhIGNvbnNpc3RlbnQgKmZ1bmN0aW9uKiAoZWFjaAoKICAgIGlucHV0IGNvbG9yIG1hcHMgdG8gZXhhY3RseSBvbmUgb3V0cHV0IGNvbG9yKSBhbmQgaXMgbm90IHRoZSBpZGVudGl0eS4KCiAgICAiIiIKCiAgICBtYXBwaW5nID0ge30KCiAgICBmb3IgZ2ksIGdvIGluIHBhaXJzOgoKICAgICAgICBpZiBncmlkcy5zaGFwZShnaSkgIT0gZ3JpZHMuc2hhcGUoZ28pOgoKICAgICAgICAgICAgcmV0dXJuIFtdCgogICAgICAgIGZvciByIGluIHJhbmdlKGdyaWRzLmhlaWdodChnaSkpOgoKICAgICAgICAgICAgZm9yIGMgaW4gcmFuZ2UoZ3JpZHMud2lkdGgoZ2kpKToKCiAgICAgICAgICAgICAgICBhLCBiID0gZ2lbcl1bY10sIGdvW3JdW2NdCgogICAgICAgICAgICAgICAgaWYgYSBpbiBtYXBwaW5nIGFuZCBtYXBwaW5nW2FdICE9IGI6CgogICAgICAgICAgICAgICAgICAgIHJldHVybiBbXQoKICAgICAgICAgICAgICAgIG1hcHBpbmdbYV0gPSBiCgogICAgaWYgbm90IG1hcHBpbmcgb3IgYWxsKGsgPT0gdiBmb3IgaywgdiBpbiBtYXBwaW5nLml0ZW1zKCkpOgoKICAgICAgICByZXR1cm4gW10KCiAgICByZXR1cm4gW1QoInJlY29sb3IiLCB0dXBsZShzb3J0ZWQobWFwcGluZy5pdGVtcygpKSkpXQoKCgoKCmRlZiBpbmZlcl9wYWQocGFpcnMpOgoKICAgICIiIklmIGV2ZXJ5IG91dHB1dCBzaGFyZXMgb25lIHNoYXBlLCBmaW5kIHRoZSAoYW5jaG9yLCBiZykgcGFkZGluZy4iIiIKCiAgICBzaGFwZXMgPSB7Z3JpZHMuc2hhcGUoZ28pIGZvciBfLCBnbyBpbiBwYWlyc30KCiAgICBpZiBsZW4oc2hhcGVzKSAhPSAxOgoKICAgICAgICByZXR1cm4gW10KCiAgICBoLCB3ID0gbmV4dChpdGVyKHNoYXBlcykpCgogICAgb3V0ID0gW10KCiAgICBmb3IgYW5jaG9yIGluIEFOQ0hPUlM6CgogICAgICAgIGZvciBiZyBpbiBDT0xPUlM6CgogICAgICAgICAgICB0ID0gVCgicGFkIiwgKGgsIHcsIGFuY2hvciwgYmcpKQoKICAgICAgICAgICAgaWYgX2NvbnNpc3RlbnQodCwgcGFpcnMpOgoKICAgICAgICAgICAgICAgIG91dC5hcHBlbmQodCkKCiAgICByZXR1cm4gb3V0CgoKCgoKZGVmIGluZmVyX3Vwc2NhbGUocGFpcnMpOgoKICAgICIiIkV4YWN0IGludGVnZXIgdXBzY2FsZSBieSBibG9jayByZXBldGl0aW9uIChiZXlvbmQgdGhlIGVudW1lcmF0ZWQgMi01KS4iIiIKCiAgICBvdXQgPSBbXQoKICAgIGZvciBmaCBpbiByYW5nZSgyLCA5KToKCiAgICAgICAgZm9yIGZ3IGluIHJhbmdlKDIsIDkpOgoKICAgICAgICAgICAgdCA9IFQoInVwIiwgKGZoLCBmdykpCgogICAgICAgICAgICBpZiBfY29uc2lzdGVudCh0LCBwYWlycyk6CgogICAgICAgICAgICAgICAgb3V0LmFwcGVuZCh0KQoKICAgIHJldHVybiBvdXQKCgoKCgpkZWYgaW5mZXJfdHJhbnNsYXRlKHBhaXJzKToKCiAgICAiIiJMYXJnZXN0LXJhbmdlIHRyYW5zbGF0aW9uIHNlYXJjaCwgd2l0aCBldmVyeSBjYW5kaWRhdGUgYmFja2dyb3VuZC4iIiIKCiAgICBvdXQgPSBbXQoKICAgIG1heF9oID0gbWF4KGdyaWRzLmhlaWdodChnaSkgZm9yIGdpLCBfIGluIHBhaXJzKQoKICAgIG1heF93ID0gbWF4KGdyaWRzLndpZHRoKGdpKSBmb3IgZ2ksIF8gaW4gcGFpcnMpCgogICAgc2VlbiA9IHNldCgpCgogICAgZm9yIGRoIGluIHJhbmdlKC1tYXhfaCwgbWF4X2ggKyAxKToKCiAgICAgICAgZm9yIGR3IGluIHJhbmdlKC1tYXhfdywgbWF4X3cgKyAxKToKCiAgICAgICAgICAgIGlmIG5vdCBkaCBhbmQgbm90IGR3OgoKICAgICAgICAgICAgICAgIGNvbnRpbnVlCgogICAgICAgICAgICBmb3IgYmcgaW4gQ09MT1JTOgoKICAgICAgICAgICAgICAgIHQgPSBUKCJ0cmFuc2xhdGUiLCAoZGgsIGR3LCBiZykpCgogICAgICAgICAgICAgICAgaWYgdC5zaWduYXR1cmUoKSBpbiBzZWVuOgoKICAgICAgICAgICAgICAgICAgICBjb250aW51ZQoKICAgICAgICAgICAgICAgIGlmIF9jb25zaXN0ZW50KHQsIHBhaXJzKToKCiAgICAgICAgICAgICAgICAgICAgb3V0LmFwcGVuZCh0KQoKICAgICAgICAgICAgICAgICAgICBzZWVuLmFkZCh0LnNpZ25hdHVyZSgpKQoKICAgIHJldHVybiBvdXQKCgoKCgpkZWYgaW5mZXJfY3JvcF9iZyhwYWlycyk6CgogICAgIiIiQ3JvcCB3aXRoIGFuIGV4cGxpY2l0IGJhY2tncm91bmQgY29sb3IgKGF1dG8tYmcgbWF5IHBpY2sgd3JvbmcpLiIiIgoKICAgIG91dCA9IFtdCgogICAgZm9yIGJnIGluIENPTE9SUzoKCiAgICAgICAgdCA9IFQoImNyb3BfYiIsIChiZywpKQoKICAgICAgICBpZiBfY29uc2lzdGVudCh0LCBwYWlycyk6CgogICAgICAgICAgICBvdXQuYXBwZW5kKHQpCgogICAgcmV0dXJuIG91dAoKCgoKCmRlZiBpbmZlcl9vYmplY3RfcmVtYXAocGFpcnMpOgoKICAgICIiIlBlci1jb2xvciBvYmplY3QgdHJhbnNmb3JtIChBR0ktMiAiZXZlcnkgY29sb3IgbW92ZXMvcmVjb2xvcnMgYnkgaXRzCgogICAgb3duIHJ1bGUiIGZhbWlseSkuCgoKCiAgICBDb21wb25lbnRzIGFyZSBtYXRjaGVkIGFjcm9zcyBlYWNoIHBhaXIgYnkgbm9ybWFsaXplZCBzaGFwZTsgd2l0aGluIGEKCiAgICBzaGFwZSBncm91cCB0aGUgaW5wdXQvb3V0cHV0IG9iamVjdHMgYXJlIHBhaXJlZCBieSB0aGUgYmlqZWN0aW9uIHdpdGgKCiAgICBtaW5pbWFsIHRvdGFsIGJib3ggbW92ZW1lbnQgKHNvIGFuIGluLXBsYWNlIHJlY29sb3IgcGFpcnMgd2l0aCB6ZXJvCgogICAgbW92ZW1lbnQsIG5vdCBieSBjb2xvciBvcmRlcikuIEVhY2ggaW5wdXQgY29sb3IgbXVzdCB0aGVuIG1hcCB0byBvbmUKCiAgICAoZHIsIGRjLCBuZXdfY29sb3IpLCBpZGVudGljYWwgaW4gZXZlcnkgcGFpci4gQW55dGhpbmcgZWxzZSAoc2hhcGUKCiAgICBnYWluZWQvbG9zdCwgYW1iaWd1b3VzIHBlci1jb2xvciBydWxlKSB5aWVsZHMgbm90aGluZyDigJQgYSB3cm9uZyBndWVzcwoKICAgIHdvdWxkIGJ1cm4gb25lIG9mIHRoZSB0d28gYXR0ZW1wdHMuIENvbm5lY3Rpdml0eSA0IGlzIHRyaWVkIGZpcnN0LCB0aGVuCgogICAgOCAoZGlhZ29uYWwgbWVyZ2VzIGNhbiBhbGlnbiB0aGUgc2VnbWVudGF0aW9uKS4KCiAgICAiIiIKCiAgICBpZiBub3QgcGFpcnM6CgogICAgICAgIHJldHVybiBbXQoKICAgIGZvciBjb25uIGluICg0LCA4KToKCiAgICAgICAgZm91bmQgPSBfb2JqZWN0X3JlbWFwX3dpdGhfY29ubmVjdGl2aXR5KHBhaXJzLCBjb25uKQoKICAgICAgICBpZiBmb3VuZDoKCiAgICAgICAgICAgIHJldHVybiBmb3VuZAoKICAgIHJldHVybiBbXQoKCgoKCmRlZiBfc2hhcGVfZ3JvdXBzKG9ianMpOgoKICAgIGdyb3VwcyA9IHt9CgogICAgZm9yIG8gaW4gb2JqczoKCiAgICAgICAgZ3JvdXBzLnNldGRlZmF1bHQoby5ub3JtYWxpemVkX2NlbGxzKCksIFtdKS5hcHBlbmQobykKCiAgICByZXR1cm4gZ3JvdXBzCgoKCgoKZGVmIF9taW5fbW92ZW1lbnRfYmlqZWN0aW9ucyhpbnMsIG91dHMsIGNhcD0yNTYpOgoKICAgICIiIkFsbCBiaWplY3Rpb25zIGlucHV0LT5vdXRwdXQgYWNoaWV2aW5nIG1pbmltYWwgdG90YWwgbW92ZW1lbnQuCgoKCiAgICBFeGFjdCBmb3IgZ3JvdXBzIHVwIHRvIDcgb2JqZWN0czsgbGFyZ2VyIGdyb3VwcyBmYWxsIGJhY2sgdG8gYQoKICAgIGRldGVybWluaXN0aWMgZ3JlZWR5IG5lYXJlc3QgYXNzaWdubWVudCAob25lIGJpamVjdGlvbikuCgogICAgIiIiCgogICAgayA9IGxlbihpbnMpCgogICAgaWYgayA9PSAxOgoKICAgICAgICByZXR1cm4gWygwLCldCgoKCiAgICBkZWYgY29zdChpLCBqKToKCiAgICAgICAgcmV0dXJuIChhYnMob3V0c1tqXS5iYm94WzBdIC0gaW5zW2ldLmJib3hbMF0pCgogICAgICAgICAgICAgICAgKyBhYnMob3V0c1tqXS5iYm94WzFdIC0gaW5zW2ldLmJib3hbMV0pKQoKCgogICAgaWYgayA+IDc6CgogICAgICAgIHJlbWFpbmluZyA9IGxpc3QocmFuZ2UoaykpCgogICAgICAgIHBlcm0gPSBbTm9uZV0gKiBrCgogICAgICAgIGZvciBpIGluIHJhbmdlKGspOgoKICAgICAgICAgICAgaiA9IG1pbihyZW1haW5pbmcsIGtleT1sYW1iZGEgajogKGNvc3QoaSwgaiksIGopKQoKICAgICAgICAgICAgcGVybVtpXSA9IGoKCiAgICAgICAgICAgIHJlbWFpbmluZy5yZW1vdmUoaikKCiAgICAgICAgcmV0dXJuIFt0dXBsZShwZXJtKV0KCgoKICAgIGJlc3QgPSBOb25lCgogICAgYmlqcyA9IFtdCgogICAgZm9yIHBlcm0gaW4gaXRlcnRvb2xzLnBlcm11dGF0aW9ucyhyYW5nZShrKSk6CgogICAgICAgIGMgPSAwCgogICAgICAgIGZvciBpIGluIHJhbmdlKGspOgoKICAgICAgICAgICAgYyArPSBjb3N0KGksIHBlcm1baV0pCgogICAgICAgIGlmIGJlc3QgaXMgTm9uZSBvciBjIDwgYmVzdDoKCiAgICAgICAgICAgIGJlc3QgPSBjCgogICAgICAgICAgICBiaWpzID0gW3Blcm1dCgogICAgICAgIGVsaWYgYyA9PSBiZXN0OgoKICAgICAgICAgICAgYmlqcy5hcHBlbmQocGVybSkKCiAgICAgICAgaWYgbGVuKGJpanMpID49IGNhcDoKCiAgICAgICAgICAgIGJyZWFrCgogICAgcmV0dXJuIGJpanMKCgoKCgpkZWYgX3BhaXJfcGFydGlhbF9tYXBwaW5ncyhnaSwgZ28sIGJnX2ksIGJnX28sIGNvbm4pOgoKICAgICIiIlBlci1jb2xvciBtb3ZlL3JlY29sb3IgbWFwcyBzdWdnZXN0ZWQgYnkgT05FIHRyYWluIHBhaXIsIG9yIE5vbmUgd2hlbgoKICAgIHRoZSBwYWlyIGlzIG91dHNpZGUgdGhpcyBmYW1pbHkgKG9iamVjdCBjb3VudCBvciBzaGFwZSBtdWx0aXNldCBjaGFuZ2VkKS4KCiAgICAiIiIKCiAgICBpbl9vYmpzID0gb2JqbW9kLmNvbXBvbmVudHMoZ2ksIGNvbm4sIGJnX2kpCgogICAgb3V0X29ianMgPSBvYmptb2QuY29tcG9uZW50cyhnbywgY29ubiwgYmdfbykKCiAgICBpZiBsZW4oaW5fb2JqcykgIT0gbGVuKG91dF9vYmpzKToKCiAgICAgICAgcmV0dXJuIE5vbmUKCiAgICBpbl9ncm91cHMgPSBfc2hhcGVfZ3JvdXBzKGluX29ianMpCgogICAgb3V0X2dyb3VwcyA9IF9zaGFwZV9ncm91cHMob3V0X29ianMpCgogICAgaWYgc2V0KGluX2dyb3VwcykgIT0gc2V0KG91dF9ncm91cHMpOgoKICAgICAgICByZXR1cm4gTm9uZQoKICAgIGdyb3VwX2JpanMgPSBbXQoKICAgIGZvciBrZXkgaW4gc29ydGVkKGluX2dyb3Vwcywga2V5PXJlcHIpOgoKICAgICAgICBpbnMgPSBpbl9ncm91cHNba2V5XQoKICAgICAgICBvdXRzID0gb3V0X2dyb3Vwc1trZXldCgogICAgICAgIGlmIGxlbihpbnMpICE9IGxlbihvdXRzKToKCiAgICAgICAgICAgIHJldHVybiBOb25lCgogICAgICAgIGdyb3VwX2JpanMuYXBwZW5kKChpbnMsIG91dHMsIF9taW5fbW92ZW1lbnRfYmlqZWN0aW9ucyhpbnMsIG91dHMpKSkKCgoKICAgIHBhcnRpYWxzID0gW10KCgoKICAgIGRlZiByZWMoaSwgYWNjKToKCiAgICAgICAgaWYgbGVuKHBhcnRpYWxzKSA+PSA2NDoKCiAgICAgICAgICAgIHJldHVybgoKICAgICAgICBpZiBpID09IGxlbihncm91cF9iaWpzKToKCiAgICAgICAgICAgIHBhcnRpYWxzLmFwcGVuZChkaWN0KGFjYykpCgogICAgICAgICAgICByZXR1cm4KCiAgICAgICAgaW5zLCBvdXRzLCBiaWpzID0gZ3JvdXBfYmlqc1tpXQoKICAgICAgICBmb3IgcGVybSBpbiBiaWpzOgoKICAgICAgICAgICAgb2sgPSBUcnVlCgogICAgICAgICAgICBhZGRlZCA9IFtdCgogICAgICAgICAgICBmb3IgYSwgaiBpbiB6aXAoaW5zLCBwZXJtKToKCiAgICAgICAgICAgICAgICBiID0gb3V0c1tqXQoKICAgICAgICAgICAgICAgIGVudHJ5ID0gKGIuYmJveFswXSAtIGEuYmJveFswXSwgYi5iYm94WzFdIC0gYS5iYm94WzFdLAoKICAgICAgICAgICAgICAgICAgICAgICAgIGIuY29sb3IpCgogICAgICAgICAgICAgICAgaWYgYS5jb2xvciBpbiBhY2MgYW5kIGFjY1thLmNvbG9yXSAhPSBlbnRyeToKCiAgICAgICAgICAgICAgICAgICAgb2sgPSBGYWxzZQoKICAgICAgICAgICAgICAgICAgICBicmVhawoKICAgICAgICAgICAgICAgIGlmIGEuY29sb3Igbm90IGluIGFjYzoKCiAgICAgICAgICAgICAgICAgICAgYWRkZWQuYXBwZW5kKGEuY29sb3IpCgogICAgICAgICAgICAgICAgYWNjW2EuY29sb3JdID0gZW50cnkKCiAgICAgICAgICAgIGlmIG9rOgoKICAgICAgICAgICAgICAgIHJlYyhpICsgMSwgYWNjKQoKICAgICAgICAgICAgZm9yIGNvbG9yIGluIGFkZGVkOgoKICAgICAgICAgICAgICAgIGFjYy5wb3AoY29sb3IsIE5vbmUpCgoKCiAgICByZWMoMCwge30pCgogICAgcmV0dXJuIHBhcnRpYWxzCgoKCgoKZGVmIF9vYmplY3RfcmVtYXBfd2l0aF9jb25uZWN0aXZpdHkocGFpcnMsIGNvbm4pOgoKICAgIGJnX291dCA9IE5vbmUKCiAgICBwYWlyX3BhcnRpYWxzID0gW10KCiAgICBmb3IgZ2ksIGdvIGluIHBhaXJzOgoKICAgICAgICBpZiBncmlkcy5zaGFwZShnaSkgIT0gZ3JpZHMuc2hhcGUoZ28pOgoKICAgICAgICAgICAgcmV0dXJuIFtdCgogICAgICAgIGJnX2kgPSBncmlkcy5iYWNrZ3JvdW5kX2NvbG9yKGdpKQoKICAgICAgICBiZ19vID0gZ3JpZHMuYmFja2dyb3VuZF9jb2xvcihnbykKCiAgICAgICAgaWYgYmdfb3V0IGlzIE5vbmU6CgogICAgICAgICAgICBiZ19vdXQgPSBiZ19vCgogICAgICAgIGVsaWYgYmdfb3V0ICE9IGJnX286CgogICAgICAgICAgICByZXR1cm4gW10KCiAgICAgICAgcGFydGlhbHMgPSBfcGFpcl9wYXJ0aWFsX21hcHBpbmdzKGdpLCBnbywgYmdfaSwgYmdfbywgY29ubikKCiAgICAgICAgaWYgbm90IHBhcnRpYWxzOgoKICAgICAgICAgICAgcmV0dXJuIFtdCgogICAgICAgIHBhaXJfcGFydGlhbHMuYXBwZW5kKHBhcnRpYWxzKQoKCgogICAgY2FuZGlkYXRlcyA9IGxpc3QocGFpcl9wYXJ0aWFsc1swXSkKCiAgICBmb3IgcGFydGlhbHMgaW4gcGFpcl9wYXJ0aWFsc1sxOl06CgogICAgICAgIG54dCA9IHt9CgogICAgICAgIGZvciBjYW5kIGluIGNhbmRpZGF0ZXM6CgogICAgICAgICAgICBmb3IgcCBpbiBwYXJ0aWFsczoKCiAgICAgICAgICAgICAgICBtZXJnZWQgPSBkaWN0KGNhbmQpCgogICAgICAgICAgICAgICAgb2sgPSBUcnVlCgogICAgICAgICAgICAgICAgZm9yIGNvbG9yLCBlbnRyeSBpbiBwLml0ZW1zKCk6CgogICAgICAgICAgICAgICAgICAgIGlmIGNvbG9yIGluIG1lcmdlZCBhbmQgbWVyZ2VkW2NvbG9yXSAhPSBlbnRyeToKCiAgICAgICAgICAgICAgICAgICAgICAgIG9rID0gRmFsc2UKCiAgICAgICAgICAgICAgICAgICAgICAgIGJyZWFrCgogICAgICAgICAgICAgICAgICAgIG1lcmdlZFtjb2xvcl0gPSBlbnRyeQoKICAgICAgICAgICAgICAgIGlmIG9rOgoKICAgICAgICAgICAgICAgICAgICBueHRbdHVwbGUoc29ydGVkKG1lcmdlZC5pdGVtcygpKSldID0gbWVyZ2VkCgogICAgICAgIGNhbmRpZGF0ZXMgPSBbbnh0W2tdIGZvciBrIGluIHNvcnRlZChueHQsIGtleT1yZXByKV0KCiAgICAgICAgaWYgbm90IGNhbmRpZGF0ZXM6CgogICAgICAgICAgICByZXR1cm4gW10KCgoKICAgIG91dCA9IFtdCgogICAgZm9yIG0gaW4gY2FuZGlkYXRlczoKCiAgICAgICAgaWYgYWxsKGRyID09IDAgYW5kIGRjID09IDAgYW5kIG5jID09IGMKCiAgICAgICAgICAgICAgIGZvciBjLCAoZHIsIGRjLCBuYykgaW4gbS5pdGVtcygpKToKCiAgICAgICAgICAgIGNvbnRpbnVlCgogICAgICAgIGFyZ3MgPSAoYmdfb3V0LCkgKyB0dXBsZSgKCiAgICAgICAgICAgIChjLCBkciwgZGMsIG5jKSBmb3IgYywgKGRyLCBkYywgbmMpIGluIHNvcnRlZChtLml0ZW1zKCkpKQoKICAgICAgICBvdXQuYXBwZW5kKFQoIm9ial9yZW1hcCIsIGFyZ3MpKQoKICAgIHJldHVybiBvdXQKCgoKCgpkZWYgaW5mZXJyZWRfdHJhbnNmb3JtcyhwYWlycyk6CgogICAgIiIiQWxsIGluZmVycmVkIHRyYW5zZm9ybXMsIHNpbXBsZXN0IGZpcnN0LCBkZS1kdXBsaWNhdGVkLiIiIgoKICAgIGZvdW5kID0gW10KCiAgICBmb3IgZm4gaW4gKGluZmVyX3JlY29sb3JfbWFwLCBpbmZlcl9jcm9wX2JnLCBpbmZlcl9wYWQsCgogICAgICAgICAgICAgICBpbmZlcl91cHNjYWxlLCBpbmZlcl90cmFuc2xhdGUsIGluZmVyX29iamVjdF9yZW1hcCk6CgogICAgICAgIGZvdW5kLmV4dGVuZChmbihwYWlycykpCgogICAgdW5pcSA9IHt9CgogICAgZm9yIHQgaW4gZm91bmQ6CgogICAgICAgIHVuaXFbdC5zaWduYXR1cmUoKV0gPSB0CgogICAgcmV0dXJuIHNvcnRlZCh1bmlxLnZhbHVlcygpLCBrZXk9bGFtYmRhIHQ6IChsZW4odC5hcmdzKSwgdC5mYW1pbHksIHQuYXJncykp").decode("utf-8"))


In [ ]:
import base64 as _b, os as _o
_PATH = "/kaggle/working/arcagi2/solver.py"
_o.makedirs(_o.path.dirname(_PATH), exist_ok=True)
with open(_PATH, "w", encoding="utf-8") as _f:
    _f.write(_b.b64decode("IiIiVGhlIHNvbHZlcjogc2VhcmNoIHRyYW5zZm9ybSBwcm9ncmFtcyB0aGF0IHJlcHJvZHVjZSBldmVyeSB0cmFpbiBwYWlyLgoKCgpTdHJhdGVneQoKLS0tLS0tLS0KCjEuIEluZmVyIGhpZ2gtcHJlY2lzaW9uIHRyYW5zZm9ybXMgZnJvbSB0aGUgZGVtb25zdHJhdGlvbnMgKHJlY29sb3IgbWFwcywKCiAgIHBhZGRpbmcgdGFyZ2V0cywgdHJhbnNsYXRpb25zKSDigJQgdGhlc2UgYXJlIGRlcml2ZWQsIG5vdCBndWVzc2VkLgoKMi4gRW51bWVyYXRlIHRoZSBjbG9zZWQgdHJhbnNmb3JtIGxpYnJhcnkgYW5kIGtlZXAgZXZlcnl0aGluZyB0aGF0CgogICByZXByb2R1Y2VzIEFMTCB0cmFpbiBwYWlycyBleGFjdGx5IChkZXB0aCAxKS4KCjMuIElmIG5vdGhpbmcgaXMgY29uc2lzdGVudCwgc2VhcmNoIGRlcHRoLTIgY29tcG9zaXRpb25zIChhbmQgYSBiZWFtLWxpbWl0ZWQKCiAgIGRlcHRoIDMpIHdpdGggcGFpci0wLWZpcnN0IHJlamVjdGlvbiBvcmRlcmluZy4KCjQuIFByb2R1Y2UgVFdPIGF0dGVtcHRzIHBlciB0ZXN0IGlucHV0LCBiZWNhdXNlIEthZ2dsZSBzY29yZXMgYSB0YXNrIG91dHB1dAoKICAgYXMgY29ycmVjdCB3aGVuIGVpdGhlciBhdHRlbXB0IG1hdGNoZXMuIEF0dGVtcHQgMSBpcyB0aGUgc2ltcGxlc3QKCiAgIGNvbnNpc3RlbnQgcHJvZ3JhbTsgYXR0ZW1wdCAyIGlzIHRoZSBzaW1wbGVzdCBwcm9ncmFtIHdpdGggYSBkaWZmZXJlbnQKCiAgIGZhbWlseSBzaWduYXR1cmUgKGdlbnVpbmUgZGl2ZXJzaXR5LCBub3Qgbm9pc2UpLgoKCgpFdmVyeSBjb25zaXN0ZW5jeSBjaGVjayBpcyBleGFjdCBlcXVhbGl0eSBvbiB0aGUgZGVtb25zdHJhdGlvbiBwYWlycy4gVGhlCgpzb2x2ZXIgbmV2ZXIgInZvdGVzIiBvciBhdmVyYWdlcyDigJQgYSBwcm9ncmFtIGVpdGhlciByZXByb2R1Y2VzIHRoZQoKZGVtb25zdHJhdGlvbnMgb3IgaXQgaXMgZGlzY2FyZGVkLgoKIiIiCgoKCmltcG9ydCB0aW1lCgoKCmZyb20gLiBpbXBvcnQgZ3JpZHMKCmZyb20gLiBpbXBvcnQgZHNsCgpmcm9tIC4gaW1wb3J0IGluZmVyCgpmcm9tIC5kc2wgaW1wb3J0IFQKCgoKCgpERUZBVUxUX0JVREdFVF9TID0gOC4wCgpERUZBVUxUX01BWF9ERVBUSCA9IDMKCkJFQU1fV0lEVEggPSA0MDAwCgoKCgoKZGVmIF9jb25zaXN0ZW50KHQsIHBhaXJzKToKCiAgICBmb3IgZ2ksIGdvIGluIHBhaXJzOgoKICAgICAgICBpZiB0LmFwcGx5KGdpKSAhPSBnbzoKCiAgICAgICAgICAgIHJldHVybiBGYWxzZQoKICAgIHJldHVybiBUcnVlCgoKCgoKZGVmIF9wcm9ncmFtX2NvbnNpc3RlbnQocHJvZ3JhbSwgcGFpcnMpOgoKICAgIGZvciBnaSwgZ28gaW4gcGFpcnM6CgogICAgICAgIGlmIGRzbC5hcHBseV9wcm9ncmFtKGdpLCBwcm9ncmFtKSAhPSBnbzoKCiAgICAgICAgICAgIHJldHVybiBGYWxzZQoKICAgIHJldHVybiBUcnVlCgoKCgoKZGVmIF9yYW5rZWQoc29sdXRpb25zKToKCiAgICAiIiJEZS1kdXBsaWNhdGUgYnkgc2lnbmF0dXJlLCBzaW1wbGVzdCBmaXJzdC4iIiIKCiAgICB1bmlxID0ge30KCiAgICBmb3IgcCBpbiBzb2x1dGlvbnM6CgogICAgICAgIHVuaXFbdHVwbGUodC5zaWduYXR1cmUoKSBmb3IgdCBpbiBwKV0gPSBwCgogICAgcmV0dXJuIHNvcnRlZCh1bmlxLnZhbHVlcygpLCBrZXk9ZHNsLnByb2dyYW1fY29zdCkKCgoKCgpkZWYgc2VhcmNoX3Byb2dyYW1zKHBhaXJzLCBidWRnZXRfcz1ERUZBVUxUX0JVREdFVF9TLAoKICAgICAgICAgICAgICAgICAgICBtYXhfZGVwdGg9REVGQVVMVF9NQVhfREVQVEgsIGRlYWRsaW5lPU5vbmUpOgoKICAgICIiIlJldHVybiBjb25zaXN0ZW50IHByb2dyYW1zLCBzaW1wbGVzdCBmaXJzdCAobWF5IGJlIGVtcHR5KS4iIiIKCiAgICBpZiBkZWFkbGluZSBpcyBOb25lOgoKICAgICAgICBkZWFkbGluZSA9IHRpbWUubW9ub3RvbmljKCkgKyBidWRnZXRfcwoKCgogICAgY2FuZHMgPSBkc2wuZW51bWVyYXRlZF9jYW5kaWRhdGVzKCkKCiAgICBjYW5kcy5leHRlbmQoaW5mZXIuaW5mZXJyZWRfdHJhbnNmb3JtcyhwYWlycykpCgogICAgc2VlbiA9IHNldCgpCgogICAgdW5pcSA9IFtdCgogICAgZm9yIHQgaW4gY2FuZHM6CgogICAgICAgIGlmIHQuc2lnbmF0dXJlKCkgbm90IGluIHNlZW46CgogICAgICAgICAgICBzZWVuLmFkZCh0LnNpZ25hdHVyZSgpKQoKICAgICAgICAgICAgdW5pcS5hcHBlbmQodCkKCiAgICBjYW5kcyA9IHVuaXEKCiAgICBzdGFnZTIgPSBkc2wuc2Vjb25kX3N0YWdlX2NhbmRpZGF0ZXMoKQoKCgogICAgc29sdXRpb25zID0gW10KCgoKICAgICMgLS0tLSBkZXB0aCAxIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KCiAgICBmb3IgdCBpbiBjYW5kczoKCiAgICAgICAgaWYgX2NvbnNpc3RlbnQodCwgcGFpcnMpOgoKICAgICAgICAgICAgc29sdXRpb25zLmFwcGVuZCgodCwpKQoKICAgIGlmIHNvbHV0aW9uczoKCiAgICAgICAgcmV0dXJuIF9yYW5rZWQoc29sdXRpb25zKQoKCgogICAgaWYgbWF4X2RlcHRoIDwgMiBvciB0aW1lLm1vbm90b25pYygpID4gZGVhZGxpbmU6CgogICAgICAgIHJldHVybiBbXQoKCgogICAgIyAtLS0tIGRlcHRoIDIgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQoKICAgIHBhcnRpYWxzID0gW10KCiAgICBmb3IgdDEgaW4gY2FuZHM6CgogICAgICAgIGlmIHRpbWUubW9ub3RvbmljKCkgPiBkZWFkbGluZToKCiAgICAgICAgICAgIGJyZWFrCgogICAgICAgIGludGVycyA9IFt0MS5hcHBseShnaSkgZm9yIGdpLCBfIGluIHBhaXJzXQoKICAgICAgICBmb3IgdDIgaW4gc3RhZ2UyOgoKICAgICAgICAgICAgaWYgdDIuYXBwbHkoaW50ZXJzWzBdKSAhPSBwYWlyc1swXVsxXToKCiAgICAgICAgICAgICAgICBjb250aW51ZQoKICAgICAgICAgICAgaWYgYWxsKHQyLmFwcGx5KGludGVyc1tpXSkgPT0gcGFpcnNbaV1bMV0KCiAgICAgICAgICAgICAgICAgICBmb3IgaSBpbiByYW5nZSgxLCBsZW4ocGFpcnMpKSk6CgogICAgICAgICAgICAgICAgc29sdXRpb25zLmFwcGVuZCgodDEsIHQyKSkKCiAgICAgICAgICAgIGVsc2U6CgogICAgICAgICAgICAgICAgcGFydGlhbHMuYXBwZW5kKCh0MSwgdDIsIFt0Mi5hcHBseSh4KSBmb3IgeCBpbiBpbnRlcnNdKSkKCiAgICBpZiBzb2x1dGlvbnM6CgogICAgICAgIHJldHVybiBfcmFua2VkKHNvbHV0aW9ucykKCgoKICAgIGlmIG1heF9kZXB0aCA8IDMgb3IgdGltZS5tb25vdG9uaWMoKSA+IGRlYWRsaW5lOgoKICAgICAgICByZXR1cm4gW10KCgoKICAgICMgLS0tLSBkZXB0aCAzIChiZWFtIG92ZXIgcGFydGlhbHMgdGhhdCBhbHJlYWR5IHNhdGlzZnkgcGFpciAwKSAtLS0tLS0tCgogICAgcGFydGlhbHMuc29ydChrZXk9bGFtYmRhIHA6IGRzbC5wcm9ncmFtX2Nvc3QocFs6Ml0pKQoKICAgIGZvciB0MSwgdDIsIGludGVycyBpbiBwYXJ0aWFsc1s6QkVBTV9XSURUSF06CgogICAgICAgIGZvciB0MyBpbiBzdGFnZTI6CgogICAgICAgICAgICBpZiB0My5hcHBseShpbnRlcnNbMF0pICE9IHBhaXJzWzBdWzFdOgoKICAgICAgICAgICAgICAgIGNvbnRpbnVlCgogICAgICAgICAgICBpZiBhbGwodDMuYXBwbHkoaW50ZXJzW2ldKSA9PSBwYWlyc1tpXVsxXQoKICAgICAgICAgICAgICAgICAgIGZvciBpIGluIHJhbmdlKDEsIGxlbihwYWlycykpKToKCiAgICAgICAgICAgICAgICBzb2x1dGlvbnMuYXBwZW5kKCh0MSwgdDIsIHQzKSkKCiAgICAgICAgaWYgdGltZS5tb25vdG9uaWMoKSA+IGRlYWRsaW5lOgoKICAgICAgICAgICAgYnJlYWsKCiAgICByZXR1cm4gX3JhbmtlZChzb2x1dGlvbnMpCgoKCgoKZGVmIF9mYWxsYmFja19hdHRlbXB0cyhwYWlycywgdGVzdF9pbnB1dCk6CgogICAgIiIiTm8gY29uc2lzdGVudCBwcm9ncmFtOiB0d28gY2hlYXAsIGRpdmVyc2UgZ3Vlc3Nlcy4iIiIKCiAgICBhMSA9IHRlc3RfaW5wdXQgICMgaWRlbnRpdHk6IHNvbWUgdGFza3MgYXJlIChuZWFyKSBpZGVudGl0eQoKICAgIHNoYXBlcyA9IFtncmlkcy5zaGFwZShnbykgZm9yIF8sIGdvIGluIHBhaXJzXQoKICAgIGNvbG9ycyA9IFtdCgogICAgZm9yIF8sIGdvIGluIHBhaXJzOgoKICAgICAgICBjb2xvcnMuZXh0ZW5kKGdyaWRzLmNvbG9yc191c2VkKGdvKSkKCiAgICBmcm9tIGNvbGxlY3Rpb25zIGltcG9ydCBDb3VudGVyCgogICAgbW9kYWxfc2hhcGUgPSBDb3VudGVyKHNoYXBlcykubW9zdF9jb21tb24oMSlbMF1bMF0KCiAgICBtb2RhbF9jb2xvciA9IENvdW50ZXIoY29sb3JzKS5tb3N0X2NvbW1vbigxKVswXVswXSBpZiBjb2xvcnMgZWxzZSAwCgogICAgYTIgPSBncmlkcy5ibGFuayhtb2RhbF9zaGFwZVswXSwgbW9kYWxfc2hhcGVbMV0sIG1vZGFsX2NvbG9yKQoKICAgIHJldHVybiBhMSwgYTIKCgoKCgpkZWYgc29sdmVfdGFzayh0YXNrLCBidWRnZXRfcz1ERUZBVUxUX0JVREdFVF9TLAoKICAgICAgICAgICAgICAgbWF4X2RlcHRoPURFRkFVTFRfTUFYX0RFUFRIKToKCiAgICAiIiJTb2x2ZSBvbmUgdGFzayBkaWN0IChhcyBsb2FkZWQgZnJvbSB0aGUgQVJDLUFHSS0yIEpTT04gZm9ybWF0KS4KCgoKICAgIFJldHVybnMgeyJwcmVkaWN0aW9ucyI6IFsgeyJhdHRlbXB0XzEiOiBncmlkLCAiYXR0ZW1wdF8yIjogZ3JpZH0sIC4uLiBdLAoKICAgICAgICAgICAgICJwcm9ncmFtIjogW3RyYW5zZm9ybSBzaWduYXR1cmVzXSBvciBOb25lLAoKICAgICAgICAgICAgICJzb2x2ZWQiOiBib29sfQoKICAgICIiIgoKICAgIHBhaXJzID0gWyhncmlkcy5mcm9tX3Jvd3MocFsiaW5wdXQiXSksIGdyaWRzLmZyb21fcm93cyhwWyJvdXRwdXQiXSkpCgogICAgICAgICAgICAgZm9yIHAgaW4gdGFza1sidHJhaW4iXV0KCiAgICB0ZXN0X2lucHV0cyA9IFtncmlkcy5mcm9tX3Jvd3MocFsiaW5wdXQiXSkgZm9yIHAgaW4gdGFza1sidGVzdCJdXQoKCgogICAgcHJvZ3JhbXMgPSBzZWFyY2hfcHJvZ3JhbXMocGFpcnMsIGJ1ZGdldF9zPWJ1ZGdldF9zLAoKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIG1heF9kZXB0aD1tYXhfZGVwdGgpCgoKCiAgICBwcmVkaWN0aW9ucyA9IFtdCgogICAgdXNlZCA9IE5vbmUKCiAgICBpZiBwcm9ncmFtczoKCiAgICAgICAgdXNlZCA9IHByb2dyYW1zWzBdCgogICAgICAgIGFsdCA9IE5vbmUKCiAgICAgICAgc2lnMCA9IHR1cGxlKHQuZmFtaWx5IGZvciB0IGluIHVzZWQpCgogICAgICAgIGZvciBwIGluIHByb2dyYW1zWzE6XToKCiAgICAgICAgICAgIGlmIHR1cGxlKHQuZmFtaWx5IGZvciB0IGluIHApICE9IHNpZzA6CgogICAgICAgICAgICAgICAgYWx0ID0gcAoKICAgICAgICAgICAgICAgIGJyZWFrCgogICAgICAgIGZvciBnaSBpbiB0ZXN0X2lucHV0czoKCiAgICAgICAgICAgIGExID0gZHNsLmFwcGx5X3Byb2dyYW0oZ2ksIHVzZWQpCgogICAgICAgICAgICBhMiA9IGRzbC5hcHBseV9wcm9ncmFtKGdpLCBhbHQpIGlmIGFsdCBlbHNlIF9mYWxsYmFja19hdHRlbXB0cygKCiAgICAgICAgICAgICAgICBwYWlycywgZ2kpWzFdCgogICAgICAgICAgICBwcmVkaWN0aW9ucy5hcHBlbmQoeyJhdHRlbXB0XzEiOiBhMSwgImF0dGVtcHRfMiI6IGEyfSkKCiAgICAgICAgcmV0dXJuIHsicHJlZGljdGlvbnMiOiBwcmVkaWN0aW9ucywKCiAgICAgICAgICAgICAgICAicHJvZ3JhbSI6IFt0LnNpZ25hdHVyZSgpIGZvciB0IGluIHVzZWRdLAoKICAgICAgICAgICAgICAgICJzb2x2ZWQiOiBUcnVlfQoKCgogICAgZm9yIGdpIGluIHRlc3RfaW5wdXRzOgoKICAgICAgICBhMSwgYTIgPSBfZmFsbGJhY2tfYXR0ZW1wdHMocGFpcnMsIGdpKQoKICAgICAgICBwcmVkaWN0aW9ucy5hcHBlbmQoeyJhdHRlbXB0XzEiOiBhMSwgImF0dGVtcHRfMiI6IGEyfSkKCiAgICByZXR1cm4geyJwcmVkaWN0aW9ucyI6IHByZWRpY3Rpb25zLCAicHJvZ3JhbSI6IE5vbmUsICJzb2x2ZWQiOiBGYWxzZX0=").decode("utf-8"))


In [ ]:
import base64 as _b, os as _o
_PATH = "/kaggle/working/arcagi2/taskio.py"
_o.makedirs(_o.path.dirname(_PATH), exist_ok=True)
with open(_PATH, "w", encoding="utf-8") as _f:
    _f.write(_b.b64decode("IiIiVGFzayBJL08sIHNjb3JpbmcsIGFuZCBzdWJtaXNzaW9uIHNlcmlhbGl6YXRpb24gZm9yIEFSQy1BR0ktMi4KCgoKU2NvcmluZyBmb2xsb3dzIHRoZSBvZmZpY2lhbCBydWxlIHZlcmJhdGltOiBhIHRhc2sgb3V0cHV0IHNjb3JlcyAxIHdoZW4KCkVJVEhFUiBhdHRlbXB0IG1hdGNoZXMgdGhlIGdyb3VuZCB0cnV0aCBleGFjdGx5OyB0aGUgZmluYWwgc2NvcmUgaXMgdGhlCgphdmVyYWdlIG92ZXIgYWxsIHRhc2sgdGVzdCBvdXRwdXRzLgoKIiIiCgoKCmltcG9ydCBnbG9iCgppbXBvcnQganNvbgoKaW1wb3J0IG9zCgoKCmZyb20gLiBpbXBvcnQgZ3JpZHMKCmZyb20gLiBpbXBvcnQgc29sdmVyIGFzIHNvbHZlcm1vZAoKCgoKCmRlZiBsb2FkX3Rhc2socGF0aCk6CgogICAgd2l0aCBvcGVuKHBhdGgsICJyIiwgZW5jb2Rpbmc9InV0Zi04IikgYXMgZmg6CgogICAgICAgIHJldHVybiBqc29uLmxvYWQoZmgpCgoKCgoKZGVmIGxvYWRfZGF0YXNldChkaXJlY3RvcnkpOgoKICAgICIiInRhc2tfaWQgLT4gdGFzayBkaWN0LCBmb3IgZXZlcnkgKi5qc29uIGluIHRoZSBkaXJlY3RvcnkuIiIiCgogICAgb3V0ID0ge30KCiAgICBmb3IgcGF0aCBpbiBzb3J0ZWQoZ2xvYi5nbG9iKG9zLnBhdGguam9pbihkaXJlY3RvcnksICIqLmpzb24iKSkpOgoKICAgICAgICB0aWQgPSBvcy5wYXRoLnNwbGl0ZXh0KG9zLnBhdGguYmFzZW5hbWUocGF0aCkpWzBdCgogICAgICAgIG91dFt0aWRdID0gbG9hZF90YXNrKHBhdGgpCgogICAgcmV0dXJuIG91dAoKCgoKCmRlZiBzb2x2ZV9kYXRhc2V0KGRhdGFzZXQsIGJ1ZGdldF9zPXNvbHZlcm1vZC5ERUZBVUxUX0JVREdFVF9TLAoKICAgICAgICAgICAgICAgICAgbWF4X2RlcHRoPXNvbHZlcm1vZC5ERUZBVUxUX01BWF9ERVBUSCwgcHJvZ3Jlc3M9Tm9uZSk6CgogICAgIiIiU29sdmUgZXZlcnkgdGFzay4gUmV0dXJucyB7dGFza19pZDogc29sdmVfdGFzayByZXN1bHR9LiIiIgoKICAgIHJlc3VsdHMgPSB7fQoKICAgIGZvciBpLCAodGlkLCB0YXNrKSBpbiBlbnVtZXJhdGUoc29ydGVkKGRhdGFzZXQuaXRlbXMoKSkpOgoKICAgICAgICByZXN1bHRzW3RpZF0gPSBzb2x2ZXJtb2Quc29sdmVfdGFzayh0YXNrLCBidWRnZXRfcz1idWRnZXRfcywKCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgbWF4X2RlcHRoPW1heF9kZXB0aCkKCiAgICAgICAgaWYgcHJvZ3Jlc3M6CgogICAgICAgICAgICBwcm9ncmVzcyhpICsgMSwgbGVuKGRhdGFzZXQpLCB0aWQsIHJlc3VsdHNbdGlkXSkKCiAgICByZXR1cm4gcmVzdWx0cwoKCgoKCmRlZiBzY29yZV90YXNrKHRhc2ssIHJlc3VsdCk6CgogICAgIiIiKG5fY29ycmVjdCwgbl90b3RhbCwgYXR0ZW1wdDFfY29ycmVjdCwgZWl0aGVyX2NvcnJlY3QpIGZvciBvbmUgdGFzay4iIiIKCiAgICB0b3RhbCA9IDAKCiAgICBlaXRoZXIgPSAwCgogICAgb25seTEgPSAwCgogICAgZm9yIGksIHByZWQgaW4gZW51bWVyYXRlKHJlc3VsdFsicHJlZGljdGlvbnMiXSk6CgogICAgICAgIGd0ID0gZ3JpZHMuZnJvbV9yb3dzKHRhc2tbInRlc3QiXVtpXVsib3V0cHV0Il0pCgogICAgICAgIHRvdGFsICs9IDEKCiAgICAgICAgYTEgPSBwcmVkWyJhdHRlbXB0XzEiXSA9PSBndAoKICAgICAgICBhMiA9IHByZWRbImF0dGVtcHRfMiJdID09IGd0CgogICAgICAgIGlmIGExIG9yIGEyOgoKICAgICAgICAgICAgZWl0aGVyICs9IDEKCiAgICAgICAgaWYgYTE6CgogICAgICAgICAgICBvbmx5MSArPSAxCgogICAgcmV0dXJuIHRvdGFsLCBlaXRoZXIsIG9ubHkxLCBlaXRoZXIKCgoKCgpkZWYgc2NvcmVfZGF0YXNldChkYXRhc2V0LCByZXN1bHRzKToKCiAgICAiIiJBZ2dyZWdhdGUgdGhlIG9mZmljaWFsIGVpdGhlci1hdHRlbXB0IGFjY3VyYWN5IG92ZXIgYSBkYXRhc2V0LiIiIgoKICAgIHRvdGFsID0gMAoKICAgIGVpdGhlciA9IDAKCiAgICBvbmx5MSA9IDAKCiAgICBzb2x2ZWRfdGFza3MgPSAwCgogICAgZm9yIHRpZCwgdGFzayBpbiBkYXRhc2V0Lml0ZW1zKCk6CgogICAgICAgIHJlcyA9IHJlc3VsdHNbdGlkXQoKICAgICAgICB0LCBlLCBvLCBfID0gc2NvcmVfdGFzayh0YXNrLCByZXMpCgogICAgICAgIHRvdGFsICs9IHQKCiAgICAgICAgZWl0aGVyICs9IGUKCiAgICAgICAgb25seTEgKz0gbwoKICAgICAgICBpZiByZXMuZ2V0KCJzb2x2ZWQiKToKCiAgICAgICAgICAgIHNvbHZlZF90YXNrcyArPSAxCgogICAgcmV0dXJuIHsKCiAgICAgICAgInRhc2tzIjogbGVuKGRhdGFzZXQpLAoKICAgICAgICAidGVzdF9vdXRwdXRzIjogdG90YWwsCgogICAgICAgICJlaXRoZXJfY29ycmVjdCI6IGVpdGhlciwKCiAgICAgICAgImF0dGVtcHQxX2NvcnJlY3QiOiBvbmx5MSwKCiAgICAgICAgImFjY3VyYWN5IjogKGVpdGhlciAvIHRvdGFsKSBpZiB0b3RhbCBlbHNlIDAuMCwKCiAgICAgICAgImF0dGVtcHQxX2FjY3VyYWN5IjogKG9ubHkxIC8gdG90YWwpIGlmIHRvdGFsIGVsc2UgMC4wLAoKICAgICAgICAicHJvZ3JhbV9mb3VuZF90YXNrcyI6IHNvbHZlZF90YXNrcywKCiAgICB9CgoKCgoKZGVmIGJ1aWxkX3N1Ym1pc3Npb24oZGF0YXNldCwgcmVzdWx0cyk6CgogICAgIiIiVGhlIEthZ2dsZSBzdWJtaXNzaW9uLmpzb24gc3RydWN0dXJlLiIiIgoKICAgIHN1YiA9IHt9CgogICAgZm9yIHRpZCwgdGFzayBpbiBzb3J0ZWQoZGF0YXNldC5pdGVtcygpKToKCiAgICAgICAgZW50cmllcyA9IFtdCgogICAgICAgIGZvciBwcmVkIGluIHJlc3VsdHNbdGlkXVsicHJlZGljdGlvbnMiXToKCiAgICAgICAgICAgIGVudHJpZXMuYXBwZW5kKHsKCiAgICAgICAgICAgICAgICAiYXR0ZW1wdF8xIjogZ3JpZHMudG9fcm93cyhwcmVkWyJhdHRlbXB0XzEiXSksCgogICAgICAgICAgICAgICAgImF0dGVtcHRfMiI6IGdyaWRzLnRvX3Jvd3MocHJlZFsiYXR0ZW1wdF8yIl0pLAoKICAgICAgICAgICAgfSkKCiAgICAgICAgc3ViW3RpZF0gPSBlbnRyaWVzCgogICAgcmV0dXJuIHN1YgoKCgoKCmRlZiB3cml0ZV9zdWJtaXNzaW9uKHN1YiwgcGF0aCk6CgogICAgd2l0aCBvcGVuKHBhdGgsICJ3IiwgZW5jb2Rpbmc9InV0Zi04IikgYXMgZmg6CgogICAgICAgIGpzb24uZHVtcChzdWIsIGZoKQoKCgoKCmRlZiB2YWxpZGF0ZV9zdWJtaXNzaW9uKHN1YiwgZGF0YXNldCk6CgogICAgIiIiU3RydWN0dXJhbCBnYXRlOiBldmVyeSB0YXNrIGlkIHByZXNlbnQsIGV2ZXJ5IGF0dGVtcHQgYSBncmlkLiIiIgoKICAgIHByb2JsZW1zID0gW10KCiAgICBmb3IgdGlkLCB0YXNrIGluIGRhdGFzZXQuaXRlbXMoKToKCiAgICAgICAgaWYgdGlkIG5vdCBpbiBzdWI6CgogICAgICAgICAgICBwcm9ibGVtcy5hcHBlbmQoIm1pc3NpbmcgdGFzayAlcyIgJSB0aWQpCgogICAgICAgICAgICBjb250aW51ZQoKICAgICAgICBlbnRyaWVzID0gc3ViW3RpZF0KCiAgICAgICAgaWYgbGVuKGVudHJpZXMpICE9IGxlbih0YXNrWyJ0ZXN0Il0pOgoKICAgICAgICAgICAgcHJvYmxlbXMuYXBwZW5kKCJ0YXNrICVzOiAlZCBlbnRyaWVzIGZvciAlZCB0ZXN0IGlucHV0cyIKCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAlICh0aWQsIGxlbihlbnRyaWVzKSwgbGVuKHRhc2tbInRlc3QiXSkpKQoKICAgICAgICAgICAgY29udGludWUKCiAgICAgICAgZm9yIGksIGVudHJ5IGluIGVudW1lcmF0ZShlbnRyaWVzKToKCiAgICAgICAgICAgIGZvciBrZXkgaW4gKCJhdHRlbXB0XzEiLCAiYXR0ZW1wdF8yIik6CgogICAgICAgICAgICAgICAgaWYga2V5IG5vdCBpbiBlbnRyeToKCiAgICAgICAgICAgICAgICAgICAgcHJvYmxlbXMuYXBwZW5kKCJ0YXNrICVzWyVkXTogbWlzc2luZyAlcyIgJSAodGlkLCBpLCBrZXkpKQoKICAgICAgICAgICAgICAgICAgICBjb250aW51ZQoKICAgICAgICAgICAgICAgIGcgPSBlbnRyeVtrZXldCgogICAgICAgICAgICAgICAgaWYgbm90IGcgb3Igbm90IGFsbChpc2luc3RhbmNlKHIsIGxpc3QpIGZvciByIGluIGcpOgoKICAgICAgICAgICAgICAgICAgICBwcm9ibGVtcy5hcHBlbmQoInRhc2sgJXNbJWRdLiVzOiBub3QgYSBncmlkIiAlICh0aWQsIGksIGtleSkpCgogICAgcmV0dXJuIHByb2JsZW1z").decode("utf-8"))


In [ ]:
import base64 as _b, os as _o
_PATH = "/kaggle/working/arcagi2/__init__.py"
_o.makedirs(_o.path.dirname(_PATH), exist_ok=True)
with open(_PATH, "w", encoding="utf-8") as _f:
    _f.write(_b.b64decode("X192ZXJzaW9uX18gPSAiMC4xLjAi").decode("utf-8"))


In [ ]:
# CELL 9 — SELF-CHECK: the embedded core must construct and solve before
# the real data is touched. If this raises, STOP: the embedding broke.
import sys
sys.path.insert(0, "/kaggle/working")

from arcagi2 import grids, dsl, solver

# 1. every family applies without raising, on a small grid
g = grids.from_rows([[0, 1, 1], [0, 1, 0], [0, 0, 0]])
for t in dsl.enumerated_candidates():
    t.apply(g)

# 2. a synthetic task with a known one-transform answer is solved exactly
task = {"train": [{"input": [[0, 1, 1], [0, 1, 0], [0, 0, 0]],
                   "output": [[0, 0, 0], [0, 1, 1], [0, 1, 0]]},
                  {"input": [[0, 2, 2], [0, 2, 0], [0, 0, 0]],
                   "output": [[0, 0, 0], [0, 2, 2], [0, 2, 0]]}],
        "test": [{"input": [[0, 3, 3], [0, 3, 0], [0, 0, 0]]}]}
res = solver.solve_task(task, budget_s=10.0, max_depth=2)
assert res["solved"], res
assert res["predictions"][0]["attempt_1"] == grids.from_rows(
    [[0, 0, 0], [0, 3, 3], [0, 3, 0]]), res["predictions"][0]
assert isinstance(res["predictions"][0]["attempt_2"], tuple)
print("mini smoke OK — %d candidates, solved: %s" % (
    len(dsl.enumerated_candidates()), res["program"]))

In [ ]:
# CELL 10 — LOAD, SOLVE, WRITE submission.json
# ✅ Verified (2026-10-18): submission.json = {task_id: [{attempt_1, attempt_2},
#    ...]} with plain lists; a test output scores when EITHER attempt matches
#    exactly. ✅ <= 12h CPU/GPU, no internet — the adaptive budget below keeps
#    the whole run inside 9.5h even on a large hidden set.
import sys, os, json, time, glob
sys.path.insert(0, "/kaggle/working")

from arcagi2 import taskio

TOTAL_BUDGET_S = 9.5 * 3600          # ✅ well inside the 12h cap
PER_TASK_CAP_S = 120.0               # depth-1 programs are found in <1s; the
                                     # cap only bounds the unsolved majority
MAX_DEPTH = 2                        # depth 1 + curated depth-2 compositions

def discover_tasks():
    '''Every task JSON under /kaggle/input.

    Layouts handled (see cell 1 output):
      single  {"train": [...], "test": [...]}           -> id from filename
      multi   {task_id: {"train": [...], "test": ...}}  -> ids from keys
      list    [ {"train": ..., "test": ...}, ... ]      -> ids from position
    'solutions' files ({task_id: [expected outputs]}) are NOT task files --
    their values are lists, not task dicts, so no branch ever matches them.
    '''
    tasks = {}
    for p in glob.glob("/kaggle/input/**/*.json", recursive=True):
        try:
            if os.path.getsize(p) > 40 * 1024 * 1024:
                continue
            with open(p, "r", encoding="utf-8") as fh:
                d = json.load(fh)
        except Exception:
            continue
        base = os.path.splitext(os.path.basename(p))[0]
        if isinstance(d, dict) and "train" in d and "test" in d:
            tasks[base] = d
        elif isinstance(d, dict) and d and all(
                isinstance(v, dict) and "train" in v and "test" in v
                for v in d.values()):
            for tid, t in d.items():
                tasks[tid] = t
        elif isinstance(d, list) and d and all(
                isinstance(x, dict) and "train" in x and "test" in x
                for x in d):
            for i, t in enumerate(d):
                tasks["%s#%d" % (base, i)] = t
    return tasks

dataset = discover_tasks()
# Bonus (2026-10-24): if the snapshot ships a solutions file
# ({task_id: [expected test outputs]}), attach the expected outputs to the
# matching test items so the local score below covers every task instead of
# only the public split. Mismatched ids or lengths are skipped, never guessed.
attached = 0
for p in glob.glob("/kaggle/input/**/*.json", recursive=True):
    try:
        with open(p, "r", encoding="utf-8") as fh:
            d = json.load(fh)
    except Exception:
        continue
    if not (isinstance(d, dict) and d
            and all(isinstance(v, list) for v in d.values())):
        continue
    for tid, outs in d.items():
        t = dataset.get(tid)
        if not t or not isinstance(outs, list) or len(outs) != len(t["test"]):
            continue
        for e, o in zip(t["test"], outs):
            if "output" not in e:
                e["output"] = o
                attached += 1
print("attached expected outputs:", attached)
print("tasks discovered:", len(dataset))
assert dataset, "no tasks found — check cell 1 discovery output"

_tasks = max(1, len(dataset))
# Hard guarantee: per_task * _tasks <= TOTAL_BUDGET_S, so the whole run stays
# inside 9.5h and therefore inside Kaggle's 12h cap even on a huge hidden set.
# The previous `max(5.0, ...)` floor could push the wall clock past 12h once
# the hidden set exceeded ~6800 tasks, which would score the run as a DNF.
per_task = min(PER_TASK_CAP_S, TOTAL_BUDGET_S / _tasks)
assert per_task * _tasks <= TOTAL_BUDGET_S + 1e-6, "budget exceeds the 9.5h cap"
print("per-task budget: %.1fs, depth: %d, tasks: %d, worst-case total: %.1fh"
      % (per_task, MAX_DEPTH, _tasks, per_task * _tasks / 3600.0))

t0 = time.time()
state = {"n": 0, "solved": 0}

def progress(i, total, tid, res):
    state["n"] = i
    state["solved"] += 1 if res.get("solved") else 0
    if i % 10 == 0 or i == total:
        print("  %d/%d done, %d solved, %.0fs elapsed"
              % (i, total, state["solved"], time.time() - t0))

results = taskio.solve_dataset(dataset, budget_s=per_task,
                               max_depth=MAX_DEPTH, progress=progress)

# score the slice we CAN score locally (public set ships test outputs)
scored = {tid: t for tid, t in dataset.items()
          if all("output" in e for e in t["test"])}
if scored:
    sc = taskio.score_dataset(scored, results)
    print("local score (public test outputs): %d/%d either-attempt correct, "
          "%d programs found" % (sc["either_correct"], sc["test_outputs"],
                                 sc["program_found_tasks"]))

sub = taskio.build_submission(dataset, results)
problems = taskio.validate_submission(sub, dataset)
assert not problems, problems[:5]
out_path = "/kaggle/working/submission.json"
taskio.write_submission(sub, out_path)
print("wrote", out_path, "tasks:", len(sub),
      "elapsed: %.0fs" % (time.time() - t0))

## Submission checklist

1. Cell 1 discovery read; the loader found every task (assert passed).
2. Cell 9 self-check printed `mini smoke OK`.
3. Cell 10 wrote `submission.json` with zero validation problems.
4. Notebook set to **public** (official: prize eligibility requires open
   source) and committed identical to
   https://github.com/Zhiyilang/arcagi2 (claim-audit rule).
5. Kaggle "Submit" -> record the **submission ID**; the Paper Track writeup
   must reference it (see `ARC_PRIZE_ACTION_CARD.md`).

Deadlines (2026-10-18 official, all 11:59 PM UTC): entry 10-26 · final
submission 11-02 · Paper Track writeup 11-09 · winners 12-04.